
# Protein Backbone Diffusion V4h

V4h returns to the best-performing V4 training objective and tests a different kind of intervention: **sampling-time guidance/projection**.

The recent V4d/V4e/V4f experiments showed that adding global/nonlocal losses during training had some leverage, but did not transfer cleanly into better free samples. V4h therefore does not add another training-time nonlocal loss. Instead, it keeps the V4 EGNN-style denoiser and local geometry objective, reloads the best checkpoint, then compares standard sampling against lightweight guided/projection sampling.

The hypothesis is inspired by conditioned/guided diffusion systems such as RFdiffusion and Chroma, but implemented as a small prototype: steer generated backbones during reverse diffusion with a conservative length-aware radius projection, and optionally repair adjacent C-alpha spacing in the low-noise part of sampling.


## What changed in V4h, and why

**One change from V4g: the noise schedule.** Everything else (the V4 EGNN denoiser, the
local geometry losses, normalization, the sampling and diagnostic code) is identical, so any
movement in the metrics is attributable to the schedule alone.

**The bug V4h fixes.** V4g used the standard linear DDPM schedule but ran it with only
`T = 100` steps. That schedule was designed for `T = 1000`. At `T = 100` the cumulative
`alpha_bar_T` is **0.364** (so `sqrt(alpha_bar_T) = 0.60`): even the *noisiest* training step
still contained ~60% of the clean structure. This is a **non-zero terminal SNR**. But every
sampler starts generation from pure `N(0, I)` noise, which the model never saw in training.
The mismatch bakes in a contraction of about `x0 ~ 0.34 * x_T` on the first reverse step:
prior Rg ~20.3 A x 0.34 = ~6.8 A, which matches the observed reverse-trajectory start of
**6.46 A**. Roughly half the global collapse is manufactured here, before the model decides
anything.

**The fix.** Switch to a **cosine schedule** (Nichol & Dhariwal 2021), which drives
`alpha_bar_T` to ~`2.4e-7` (terminal SNR ~ 0). Now pure-noise sampling matches the noisiest
training step. We keep `T = 100` so the schedule *shape* is the only variable; we do not move
to the full Lin et al. zero-terminal-SNR rescale because that requires v-prediction and
sampler changes (more moving parts than this fix needs). Cell 14 prints a terminal-SNR sanity
check and asserts `alpha_bar_T < 1e-3`.

**What to watch.** Success = mean generated Rg moves from ~8 A toward >=12-14 A, collapse
fraction drops, adjacent-CA in-band stays >=0.85. The V4g weak-projection guidance result is
retained in this notebook as a guaranteed fallback. Note: the *diagnostic* `x0_pred_rms` /
`x0_rmse` will read larger at high timesteps now (dividing by a tiny `sqrt(alpha_bar_t)`);
this is cosmetic and does not enter the optimized loss, which is governed by the noise MSE.

**Schedule toggle.** Set `BACKBONE_DIFFUSION_SCHEDULE=linear` to reproduce the old V4g
behaviour for an A/B comparison; the default is `cosine`.

---


## 1. Runtime and setup

This section detects the runtime, optionally mounts Google Drive, locates or clones the repository, resolves the CATH data directory, installs small notebook dependencies, and sets the seed/device/output directories. By default V4h writes artifacts to Google Drive when Drive mounts successfully, otherwise it falls back to the repo-local `results/v4h/` directory. Google Drive is no longer required for PyCharm or other IDE execution.


In [ ]:
import base64
import os
import random
import subprocess
import sys
import time
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from torch.utils.data import DataLoader

try:
    import google.colab  # type: ignore
    from google.colab import drive, userdata  # type: ignore
    IN_COLAB = True
except ImportError:
    drive = None  # type: ignore
    userdata = None  # type: ignore
    IN_COLAB = False

def env_flag(name: str, default: bool) -> bool:
    """Parse a boolean environment flag."""
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

DRIVE_MOUNT_MODE = os.environ.get('BACKBONE_DIFFUSION_MOUNT_DRIVE', 'auto').strip().lower()
DRIVE_MOUNT_SKIPPED = DRIVE_MOUNT_MODE in {'0', 'false', 'no', 'off', 'skip'}
SAVE_TABLE_ARTIFACTS = env_flag('BACKBONE_DIFFUSION_SAVE_TABLES', not DRIVE_MOUNT_SKIPPED)

DRIVE_MOUNTPOINT = Path('/content/drive')
DRIVE_MYDRIVE = DRIVE_MOUNTPOINT / 'MyDrive'
DRIVE_MOUNTED = False

if IN_COLAB and drive is not None and not DRIVE_MOUNT_SKIPPED:
    try:
        drive.mount(str(DRIVE_MOUNTPOINT), force_remount=False)
        DRIVE_MOUNTED = DRIVE_MYDRIVE.exists()
    except Exception as exc:  # noqa: BLE001
        if DRIVE_MOUNT_MODE in {'1', 'true', 'yes', 'on', 'required'}:
            raise RuntimeError('Google Drive mounting was explicitly requested but failed.') from exc
        print(f'Google Drive mount skipped after failure: {exc}')
else:
    print('Google Drive mount skipped.')

if DRIVE_MOUNTED:
    print(f'Drive mounted: {DRIVE_MYDRIVE}')
else:
    print('Running without Google Drive mount; artifacts default to the repo-local results directory.')

REPO_REMOTE_URL = os.environ.get('GITHUB_REPO_URL', 'https://github.com/mitsenkov/latent-structure-diffusion.git')
REPO_BRANCH = os.environ.get('GITHUB_REPO_BRANCH', 'main')
REPO_CLONE_DIR = Path(os.environ.get('REPO_CLONE_DIR', '/content/latent-structure-diffusion'))
REPO_DRIVE_DIR = Path(os.environ.get('REPO_DRIVE_DIR', '/content/drive/MyDrive/latent-structure-diffusion'))
DEFAULT_CATH_FOLDER_ID = os.environ.get('CATH_SHARED_FOLDER_ID', '')
LOCAL_DATA_CACHE = Path(os.environ.get('CATH_LOCAL_CACHE', '/content/cath_backbone_data'))
ALLOW_GIT_CLONE = os.environ.get('ALLOW_GIT_CLONE', '1' if IN_COLAB else '0') == '1'

def get_github_token() -> str | None:
    """Return a GitHub token from env, Colab userdata, or an interactive prompt."""
    token = os.environ.get('GITHUB_TOKEN')
    if token:
        return token.strip()
    if IN_COLAB:
        try:
            token = userdata.get('GITHUB_TOKEN')
        except Exception:  # noqa: BLE001
            token = None
        if token:
            return str(token).strip()
    if ALLOW_GIT_CLONE:
        token = getpass('Paste the GitHub token with read access to this repo: ').strip()
        if token:
            return token
    return None

def find_repo_root(start_paths: list[Path] | None = None) -> Path | None:
    """Return the repository root if a checkout is already present."""
    candidates = start_paths or [Path.cwd().resolve(), REPO_CLONE_DIR, REPO_DRIVE_DIR]
    for start in candidates:
        if not start.exists():
            continue
        for candidate in [start, *start.parents]:
            if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
                return candidate
    return None

def build_auth_header(token: str) -> str:
    """Build a Git basic auth header for private GitHub clone access."""
    raw = f'x-access-token:{token}'.encode('utf-8')
    encoded = base64.b64encode(raw).decode('ascii')
    return f'AUTHORIZATION: basic {encoded}'

def bootstrap_repo() -> Path:
    """Make the repo available in Colab or reuse an existing checkout."""
    existing_root = find_repo_root()
    if existing_root is not None:
      if IN_COLAB:
          token = get_github_token()
          if token:
              pull_cmd = [
                  'git',
                  '-C',
                  str(existing_root),
                  '-c',
                  f'http.extraheader={build_auth_header(token)}',
                  'pull',
                  '--ff-only',
              ]
              subprocess.run(pull_cmd, check=True, capture_output=True, text=True)
      return existing_root

    if not IN_COLAB:
        raise FileNotFoundError(
            'Could not locate the repository root. Expected a folder containing pyproject.toml and src/.'
        )

    if not ALLOW_GIT_CLONE:
        raise FileNotFoundError(
            'Could not locate a local repo checkout. In Colab, set ALLOW_GIT_CLONE=1 or define GITHUB_TOKEN.'
        )

    token = get_github_token()
    if not token:
        raise FileNotFoundError(
            'GitHub cloning was enabled, but no token was provided.'
        )

    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    clone_cmd = [
        'git',
        '-c',
        f'http.extraheader={build_auth_header(token)}',
        'clone',
        '--branch',
        REPO_BRANCH,
        REPO_REMOTE_URL,
        str(REPO_CLONE_DIR),
    ]
    try:
        subprocess.run(clone_cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as exc:
        stderr = (exc.stderr or '').strip()
        stdout = (exc.stdout or '').strip()
        details = '\n'.join(part for part in [stdout, stderr] if part)
        raise RuntimeError(
            'GitHub clone failed in this Colab runtime. The token may be missing, invalid, or lack repo read access. '            f'Command: git clone --branch {REPO_BRANCH} {REPO_REMOTE_URL} {REPO_CLONE_DIR}\n{details}'
        ) from exc
    return REPO_CLONE_DIR

def ensure_gdown_installed() -> None:
    """Install gdown on demand."""
    try:
        import gdown  # type: ignore  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gdown'])

def resolve_data_dir() -> Path:
    """Resolve the dataset directory without requiring a Drive mount."""
    env_dir = os.environ.get('CATH_DATA_DIR')
    if env_dir:
        candidate = Path(env_dir)
        if (candidate / 'chain_set.jsonl').exists() and (candidate / 'chain_set_splits.json').exists():
            return candidate

    local_candidate = LOCAL_DATA_CACHE
    if (local_candidate / 'chain_set.jsonl').exists() and (local_candidate / 'chain_set_splits.json').exists():
        return local_candidate

    if IN_COLAB:
        ensure_gdown_installed()
        import gdown  # type: ignore

        local_candidate.mkdir(parents=True, exist_ok=True)
        url = f'https://drive.google.com/drive/folders/{DEFAULT_CATH_FOLDER_ID}'
        gdown.download_folder(url=url, output=str(local_candidate), quiet=False, use_cookies=False)
        if (local_candidate / 'chain_set.jsonl').exists() and (local_candidate / 'chain_set_splits.json').exists():
            return local_candidate

    raise FileNotFoundError(
        'Could not resolve the CATH dataset directory. Set CATH_DATA_DIR to a local path with chain_set.jsonl '
        'and chain_set_splits.json, or allow the notebook to download the public shared folder.'
    )

REPO_ROOT = bootstrap_repo()
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = resolve_data_dir()

DEFAULT_ARTIFACT_DIR = (
    DRIVE_MYDRIVE / 'latent-structure-generation' / 'results'
    if DRIVE_MOUNTED
    else REPO_ROOT / 'results'
)
ARTIFACT_BASE_DIR = Path(os.environ.get('BACKBONE_DIFFUSION_ARTIFACT_BASE_DIR', str(DEFAULT_ARTIFACT_DIR)))
ARTIFACT_RUN_NAME = os.environ.get('BACKBONE_DIFFUSION_RUN_NAME', 'v4h')
ARTIFACT_DIR = ARTIFACT_BASE_DIR / ARTIFACT_RUN_NAME
FIGURE_DIR = ARTIFACT_DIR / 'figures'
TABLE_DIR = ARTIFACT_DIR / 'tables'
CHECKPOINT_DIR = ARTIFACT_DIR / 'checkpoints'
for directory in [FIGURE_DIR, TABLE_DIR, CHECKPOINT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def save_table_artifact(frame: pd.DataFrame, name: str, *, drop_columns: list[str] | None = None) -> dict[str, Path]:
    """Save a dataframe as CSV and Parquet under the run's tables directory."""
    if not SAVE_TABLE_ARTIFACTS:
        return {}
    export_frame = frame.drop(columns=[column for column in (drop_columns or []) if column in frame.columns]).copy()
    csv_path = TABLE_DIR / f'{name}.csv'
    parquet_path = TABLE_DIR / f'{name}.parquet'
    export_frame.to_csv(csv_path, index=False)
    try:
        export_frame.to_parquet(parquet_path, index=False)
    except Exception as exc:  # noqa: BLE001
        print(f'Could not write Parquet for {name}: {exc}')
    return {'csv': csv_path, 'parquet': parquet_path}

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    device_name = torch.cuda.get_device_name(0)
else:
    device_name = 'CPU'

try:
    import py3Dmol  # type: ignore
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'py3Dmol'])
    import py3Dmol  # type: ignore

if SAVE_TABLE_ARTIFACTS:
    try:
        import pyarrow  # type: ignore  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyarrow'])

from latent_structure_generation.backbone_diffusion import (
    BACKBONE_ATOMS,
    BackboneDataset,
    BackboneCoordinateEGNNDenoiser,
    BackboneNormalizationStats,
    apply_coordinate_normalisation,
    backbone_coords_to_protein,
    backbone_structure_summary,
    build_normalization_stats_from_dataframe,
    build_overlapping_backbone_chunks,
    collate_backbone_examples,
    centre_coordinates,
    create_noise_schedule,
    extract_backbone_from_coords_dict,
    flatten_backbone,
    invert_coordinate_normalisation,
    masked_coordinate_rmse,
    masked_noise_mse,
    pad_or_crop_backbone,
    predict_x0,
    q_sample,
    sample_backbone,
    sample_timesteps,
    structure_validity_report,
    to_pdb,
    unflatten_backbone,
)
from latent_structure_generation.plots import plot_ca_trace

print(f'Repository root: {REPO_ROOT}')
print(f'Data directory: {DATA_DIR}')
print(f'Artifacts: {ARTIFACT_DIR}')
print(f'Google Drive mounted: {DRIVE_MOUNTED}')
print(f'Table artifact saving enabled: {SAVE_TABLE_ARTIFACTS}')
print(f'Seed: {SEED}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Selected device: {device}')
print(f'Device name: {device_name}')
print(f'GitHub clone enabled: {ALLOW_GIT_CLONE}')
print(f'GitHub token present: {bool(os.environ.get("GITHUB_TOKEN") or (IN_COLAB and "GITHUB_TOKEN" in getattr(userdata, "keys", lambda: [])()))}')



## 2. Load the starter CATH tables

This reuses the provided Google Drive layout from the starter notebook. The split names are canonicalized to `train`, `validation`, and `test` so later code does not depend on `val` vs `validation` naming drift.


In [ ]:
chain_set_path = DATA_DIR / 'chain_set.jsonl'
split_path = DATA_DIR / 'chain_set_splits.json'

print(f'Reading {chain_set_path}')
df = pd.read_json(chain_set_path, lines=True)
print(f'Reading {split_path}')
chain_splits = pd.read_json(split_path, lines=True)

def canonical_split_name(name: str) -> str:
    """Map split labels to the canonical notebook names."""
    return {'val': 'validation', 'validation': 'validation', 'train': 'train', 'test': 'test'}.get(name, name)

split_lookup: dict[str, str] = {}

# Preserve the true train/validation/test buckets first.
for column in ['train', 'validation', 'test']:
    if column not in chain_splits.columns:
        continue
    canonical = canonical_split_name(column)
    values = chain_splits[column].iloc[0]
    for record_name in values:
        split_lookup.setdefault(record_name, canonical)

# cath_nodes is auxiliary metadata; use it only for records not already assigned.
if 'cath_nodes' in chain_splits.columns:
    cath_nodes = chain_splits['cath_nodes'].iloc[0]
    for record_name in cath_nodes.keys():
        split_lookup.setdefault(record_name, 'cath_nodes')

df['split'] = df['name'].map(split_lookup).fillna('unknown')
df['split'] = df['split'].map(canonical_split_name)

available_splits = sorted(df['split'].dropna().unique().tolist())
print('Available splits:', available_splits)
print('Split counts:')
print(df['split'].value_counts(dropna=False).sort_index())
print('Columns:', list(df.columns))
print('Example row keys:', list(df.iloc[0].index))

save_table_artifact(df, 'raw_chain_manifest', drop_columns=['coords'])


## 3. Data audit before training

This is the required review phase. It checks shapes, masks, invalid values, real lengths, truncation, coordinate ranges, backbone bond lengths, and a few visual examples before any training code runs.


In [ ]:
from collections import defaultdict

def move_batch_to_device(batch: dict[str, object], target_device: torch.device) -> dict[str, object]:
    """Move tensor values in a batch to the selected device."""
    moved: dict[str, object] = {}
    for key, value in batch.items():
        moved[key] = value.to(target_device) if torch.is_tensor(value) else value
    return moved

def audit_dataframe(split_df: pd.DataFrame, split_name: str, max_length: int) -> pd.DataFrame:
    """Audit one split and return a per-example summary table."""
    rows: list[dict[str, object]] = []
    for _, row in split_df.reset_index(drop=True).iterrows():
        record_id = row['name']
        try:
            coords, residue_mask = extract_backbone_from_coords_dict(row['coords'])
            raw_length = int(coords.shape[0])
            real_length = int(residue_mask.sum())
            coords_fixed, mask_fixed, truncated = pad_or_crop_backbone(coords, residue_mask, max_length)
            coords_t = torch.tensor(coords_fixed, dtype=torch.float32)
            mask_t = torch.tensor(mask_fixed.astype(np.float32), dtype=torch.float32)
            has_nan = bool(np.isnan(coords_fixed).any())
            has_inf = bool(np.isinf(coords_fixed).any())
            zero_real_residues = int(((np.abs(coords_fixed).sum(axis=(1, 2)) == 0.0) & mask_fixed).sum())
            summary = backbone_structure_summary(coords_t, mask_t)
            rows.append(
                {
                    'record_id': record_id,
                    'split': split_name,
                    'raw_length': raw_length,
                    'real_length': real_length,
                    'padded_length': int(mask_fixed.shape[0]),
                    'truncated': bool(truncated),
                    'has_nan': has_nan,
                    'has_inf': has_inf,
                    'zero_real_residues': zero_real_residues,
                    'keep_example': bool(real_length > 0 and not has_nan and not has_inf),
                    **summary,
                }
            )
        except Exception as exc:  # noqa: BLE001
            rows.append(
                {
                    'record_id': record_id,
                    'split': split_name,
                    'raw_length': np.nan,
                    'real_length': 0,
                    'padded_length': max_length,
                    'truncated': False,
                    'has_nan': True,
                    'has_inf': True,
                    'zero_real_residues': 0,
                    'keep_example': False,
                    'audit_error': str(exc),
                    'n_residues': 0,
                    'mean_adjacent_ca': np.nan,
                    'fraction_adjacent_ca_in_band': np.nan,
                    'mean_n_ca': np.nan,
                    'mean_ca_c': np.nan,
                    'mean_c_o': np.nan,
                    'mean_c_n': np.nan,
                    'radius_of_gyration': np.nan,
                }
            )
    return pd.DataFrame(rows)

def summarize_lengths(audit_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize real lengths, padding, and truncation for one split."""
    if audit_df.empty:
        return pd.DataFrame([
            {
                'split': 'unknown',
                'n_examples': 0,
                'min_real_length': np.nan,
                'median_real_length': np.nan,
                'max_real_length': np.nan,
                'mean_real_length': np.nan,
                'mean_padding_fraction': np.nan,
                'truncated_fraction': np.nan,
            }
        ])

    if 'keep_example' not in audit_df.columns:
        valid = audit_df.copy()
    else:
        valid = audit_df[audit_df['keep_example'].fillna(False)].copy()
    if valid.empty:
        split_name = audit_df['split'].iloc[0] if 'split' in audit_df.columns and len(audit_df) else 'unknown'
        return pd.DataFrame([
            {
                'split': split_name,
                'n_examples': 0,
                'min_real_length': np.nan,
                'median_real_length': np.nan,
                'max_real_length': np.nan,
                'mean_real_length': np.nan,
                'mean_padding_fraction': np.nan,
                'truncated_fraction': np.nan,
            }
        ])

    lengths = valid['real_length'].astype(int)
    padded_fraction = 1.0 - (lengths / valid['padded_length'].astype(int))
    return pd.DataFrame(
        [
            {
                'split': valid['split'].iloc[0],
                'n_examples': int(len(valid)),
                'min_real_length': int(lengths.min()),
                'median_real_length': float(lengths.median()),
                'max_real_length': int(lengths.max()),
                'mean_real_length': float(lengths.mean()),
                'mean_padding_fraction': float(padded_fraction.mean()),
                'truncated_fraction': float(valid['truncated'].mean()),
            }
        ]
    )

def coordinate_moments(split_df: pd.DataFrame, max_length: int) -> dict[str, np.ndarray | float]:
    """Compute mean/std/RMS statistics over real atom coordinates only."""
    sum_xyz = np.zeros(3, dtype=np.float64)
    sum_sq_xyz = np.zeros(3, dtype=np.float64)
    count = 0
    rg_values: list[float] = []
    for _, row in split_df.reset_index(drop=True).iterrows():
        coords, residue_mask = extract_backbone_from_coords_dict(row['coords'])
        if residue_mask.sum() == 0:
            continue
        coords_fixed, mask_fixed, _ = pad_or_crop_backbone(coords, residue_mask, max_length)
        coords_t = torch.tensor(coords_fixed, dtype=torch.float32)
        mask_t = torch.tensor(mask_fixed.astype(np.float32), dtype=torch.float32)
        centred = centre_coordinates(coords_t.unsqueeze(0), mask_t.unsqueeze(0))[0]
        real = centred[mask_t.bool()].reshape(-1, 3)
        sum_xyz += real.sum(dim=0).cpu().numpy()
        sum_sq_xyz += (real.pow(2)).sum(dim=0).cpu().numpy()
        count += int(real.shape[0])
        rg_values.append(float(backbone_structure_summary(coords_t, mask_t)['radius_of_gyration']))
    mean = sum_xyz / max(count, 1)
    var = sum_sq_xyz / max(count, 1) - mean**2
    std = np.sqrt(np.clip(var, 1e-6, None))
    return {'mean': mean, 'std': std, 'rms': float(np.sqrt(np.mean(sum_sq_xyz / max(count, 1)))), 'count': count, 'radius_of_gyration_mean': float(np.mean(rg_values)) if rg_values else float('nan')}

MAX_SEQ_LENGTH = 256
train_audit = audit_dataframe(df[df['split'] == 'train'], 'train', MAX_SEQ_LENGTH)
val_audit = audit_dataframe(df[df['split'] == 'validation'], 'validation', MAX_SEQ_LENGTH)
test_audit = audit_dataframe(df[df['split'] == 'test'], 'test', MAX_SEQ_LENGTH)

audit_df = pd.concat([train_audit, val_audit, test_audit], ignore_index=True)

length_summary_df = pd.concat(
    [summarize_lengths(train_audit), summarize_lengths(val_audit), summarize_lengths(test_audit)],
    ignore_index=True,
)

print('Audit summary by split:')
display(length_summary_df)

print('Invalid / filtered examples by split:')
filter_summary = (
    audit_df.groupby('split', as_index=False)
    .agg(total_examples=('record_id', 'count'), kept_examples=('keep_example', 'sum'), invalid_examples=('keep_example', lambda s: int((~s).sum())), truncated_examples=('truncated', 'sum'))
)
filter_summary['filtered_out'] = filter_summary['total_examples'] - filter_summary['kept_examples']
display(filter_summary)

print('Coordinate and structure summary on the training split:')
train_moments = coordinate_moments(df[df['split'] == 'train'], MAX_SEQ_LENGTH)
print(train_moments)

print('Head of the audit table:')
display(audit_df.head(8))

save_table_artifact(train_audit, 'train_audit')
save_table_artifact(val_audit, 'validation_audit')
save_table_artifact(test_audit, 'test_audit')
save_table_artifact(audit_df, 'audit_table')
save_table_artifact(length_summary_df, 'length_summary')
save_table_artifact(filter_summary, 'filter_summary')

print('Shape checks:')
print('coords expected shape: (B, L, 4, 3)')
print('mask expected shape: (B, L)')
print('backbone atom axis:', BACKBONE_ATOMS)
print('training split examples kept:', int(train_audit['keep_example'].sum()))
print('validation split examples kept:', int(val_audit['keep_example'].sum()))
print('test split examples kept:', int(test_audit['keep_example'].sum()))


### What the audit established

The audit keeps padded positions, drops only examples with no valid backbone residues, and reports the amount of truncation caused by fixed-length batching. The same summary tables also give the train-only statistics needed for normalization. Long training chains are then expanded into overlapping fixed-length chunks so we stop throwing away so much of each protein.


In [ ]:
# Build chain-level filtered split tables first.
train_chain_df = df[df['split'] == 'train'].reset_index(drop=True)
validation_chain_df = df[df['split'] == 'validation'].reset_index(drop=True)
test_chain_df = df[df['split'] == 'test'].reset_index(drop=True)

train_df = train_chain_df[train_audit['keep_example'].values].reset_index(drop=True)
validation_df = validation_chain_df[val_audit['keep_example'].values].reset_index(drop=True)
test_df = test_chain_df[test_audit['keep_example'].values].reset_index(drop=True)

# Expand long training chains into overlapping 256-residue windows.
TRAIN_CHUNK_STRIDE = 128
train_chunked_df = build_overlapping_backbone_chunks(
    train_df,
    split='train',
    max_length=MAX_SEQ_LENGTH,
    stride=TRAIN_CHUNK_STRIDE,
    record_id_column='name',
)

print('Filtered split sizes:')
print('train:', len(train_df))
print('validation:', len(validation_df))
print('test:', len(test_df))
print('train chunks:', len(train_chunked_df))
print('unique train parent chains:', train_chunked_df['parent_chain_id'].nunique())
print('training chunk stride:', TRAIN_CHUNK_STRIDE)
chunk_count_summary = train_chunked_df.groupby('parent_chain_id').size()
print('chunk count per parent chain: min', int(chunk_count_summary.min()), 'median', float(chunk_count_summary.median()), 'max', int(chunk_count_summary.max()))
print('chunk length range:', int(train_chunked_df['chunk_length'].min()), 'to', int(train_chunked_df['chunk_length'].max()))
chunk_count_summary_df = chunk_count_summary.rename_axis('parent_chain_id').reset_index(name='chunk_count')

save_table_artifact(train_df, 'train_chains_manifest', drop_columns=['coords'])
save_table_artifact(validation_df, 'validation_chains_manifest', drop_columns=['coords'])
save_table_artifact(test_df, 'test_chains_manifest', drop_columns=['coords'])
save_table_artifact(train_chunked_df, 'train_chunked_manifest', drop_columns=['coords'])
save_table_artifact(chunk_count_summary_df, 'train_chunk_counts')

OVERFIT_DEBUG = env_flag('BACKBONE_DIFFUSION_OVERFIT_DEBUG', False)
OVERFIT_SAMPLE_COUNT = int(os.environ.get('BACKBONE_DIFFUSION_OVERFIT_SAMPLE_COUNT', '128'))
OVERFIT_BATCH_SIZE = int(os.environ.get('BACKBONE_DIFFUSION_OVERFIT_BATCH_SIZE', '32'))
OVERFIT_USE_TRAIN_SUBSET_FOR_EVAL = env_flag('BACKBONE_DIFFUSION_OVERFIT_USE_TRAIN_SUBSET_FOR_EVAL', True)
OVERFIT_SHUFFLE_TRAIN = env_flag('BACKBONE_DIFFUSION_OVERFIT_SHUFFLE_TRAIN', False)

def build_overfit_subset(dataframe: pd.DataFrame, sample_count: int) -> pd.DataFrame:
    if sample_count <= 0:
        raise ValueError('BACKBONE_DIFFUSION_OVERFIT_SAMPLE_COUNT must be positive when overfit debug is enabled.')
    sample_n = min(sample_count, len(dataframe))
    return dataframe.sample(n=sample_n, random_state=SEED).reset_index(drop=True)

train_dataset_df = train_chunked_df
validation_dataset_df = validation_df
test_dataset_df = test_df
train_record_id_column = 'chunk_id'
validation_record_id_column = 'name'
test_record_id_column = 'name'
train_loader_shuffle = True

if OVERFIT_DEBUG:
    overfit_train_subset_df = build_overfit_subset(train_chunked_df, OVERFIT_SAMPLE_COUNT)
    train_dataset_df = overfit_train_subset_df
    train_loader_shuffle = OVERFIT_SHUFFLE_TRAIN
    if OVERFIT_USE_TRAIN_SUBSET_FOR_EVAL:
        validation_dataset_df = overfit_train_subset_df.copy()
        validation_dataset_df['split'] = 'validation'
        test_dataset_df = overfit_train_subset_df.copy()
        test_dataset_df['split'] = 'test'
        validation_record_id_column = 'chunk_id'
        test_record_id_column = 'chunk_id'
    else:
        validation_dataset_df = validation_df.head(min(OVERFIT_SAMPLE_COUNT, len(validation_df))).reset_index(drop=True)
        test_dataset_df = test_df.head(min(OVERFIT_SAMPLE_COUNT, len(test_df))).reset_index(drop=True)

    print('OVERFIT DEBUG ENABLED')
    print('overfit train subset size:', len(train_dataset_df))
    print('overfit eval source:', 'train subset' if OVERFIT_USE_TRAIN_SUBSET_FOR_EVAL else 'held-out validation/test head')
    save_table_artifact(train_dataset_df, 'v4h_overfit_train_subset_manifest', drop_columns=['coords'])
    if OVERFIT_USE_TRAIN_SUBSET_FOR_EVAL:
        save_table_artifact(validation_dataset_df, 'v4h_overfit_eval_subset_manifest', drop_columns=['coords'])
else:
    print('Overfit debug disabled; using full train / validation / test datasets.')

train_dataset = BackboneDataset(train_dataset_df, split='train', max_length=MAX_SEQ_LENGTH, record_id_column=train_record_id_column)
validation_dataset = BackboneDataset(validation_dataset_df, split='validation', max_length=MAX_SEQ_LENGTH, record_id_column=validation_record_id_column)
test_dataset = BackboneDataset(test_dataset_df, split='test', max_length=MAX_SEQ_LENGTH, record_id_column=test_record_id_column)

DEFAULT_BATCH_SIZE = 32
BATCH_SIZE = min(OVERFIT_BATCH_SIZE, len(train_dataset)) if OVERFIT_DEBUG else DEFAULT_BATCH_SIZE
NUM_WORKERS = 2 if IN_COLAB else 0
PIN_MEMORY = device.type == 'cuda'

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=train_loader_shuffle,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_backbone_examples,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=min(BATCH_SIZE, len(validation_dataset)),
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_backbone_examples,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=min(BATCH_SIZE, len(test_dataset)),
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_backbone_examples,
)

print('train dataset size used by loader:', len(train_dataset))
print('validation dataset size used by loader:', len(validation_dataset))
print('test dataset size used by loader:', len(test_dataset))
print('batch size:', BATCH_SIZE)
print('train loader shuffle:', train_loader_shuffle)

example_batch = next(iter(train_loader))
print('Example batch keys:', list(example_batch.keys()))
print('coords shape:', tuple(example_batch['coords'].shape), 'dtype:', example_batch['coords'].dtype, 'device:', example_batch['coords'].device)
print('mask shape:', tuple(example_batch['mask'].shape), 'dtype:', example_batch['mask'].dtype, 'device:', example_batch['mask'].device)
print('lengths shape:', tuple(example_batch['length'].shape))
print('real_length shape:', tuple(example_batch['real_length'].shape))
print('truncated shape:', tuple(example_batch['truncated'].shape))
print('first record ids:', example_batch['record_id'][:3])
print('first parent chain ids:', example_batch['parent_chain_id'][:3])
print('first chunk ids:', example_batch['chunk_id'][:3])


## 4. Visual sanity checks on real data

This shows a few training examples before any normalization or diffusion is applied. The point is to confirm that the backbone traces and masks line up with the expected protein geometry.


In [ ]:
def plot_ca_trace_from_batch(coords: torch.Tensor, mask: torch.Tensor, title: str):
    """Plot a CA trace using the existing plotting helper."""
    ca = coords[mask.bool(), 1, :].detach().cpu().numpy()
    fig, ax = plot_ca_trace(ca, title=title)
    plt.show()
    return fig, ax

def render_backbone_example(coords: torch.Tensor, mask: torch.Tensor, title: str):
    """Render one backbone structure as a PDB-backed py3Dmol viewer."""
    protein = backbone_coords_to_protein(coords, mask)
    pdb_str = to_pdb(protein)
    view = py3Dmol.view(width=700, height=500)
    view.addModel(pdb_str, 'pdb')
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    view.zoomTo()
    view.show()
    return view

for index in range(min(2, example_batch['coords'].shape[0])):
    real_coords = example_batch['coords'][index]
    real_mask = example_batch['mask'][index]
    print(f'Real example {index}:', example_batch['record_id'][index])
    print(backbone_structure_summary(real_coords, real_mask))
    plot_ca_trace_from_batch(real_coords, real_mask.bool(), title=f"Real CA trace: {example_batch['record_id'][index]}")
    render_backbone_example(real_coords, real_mask, title=f"Real backbone: {example_batch['record_id'][index]}")


## 5. Preprocessing and train-only normalization

The model sees centered backbone coordinates that are standardized with statistics computed from the training split only. This cell also checks the flatten / unflatten / inverse-transform path before training starts.


In [ ]:
normalization_stats = build_normalization_stats_from_dataframe(train_df)
print('Normalization mean:', normalization_stats.mean)
print('Normalization std:', normalization_stats.std)

smoke_batch = next(iter(train_loader))
coords = smoke_batch['coords']
mask = smoke_batch['mask']
coords_centered = centre_coordinates(coords, mask)
coords_norm = apply_coordinate_normalisation(coords_centered, mask, normalization_stats)
coords_flat = flatten_backbone(coords_norm)
coords_roundtrip = unflatten_backbone(coords_flat)
coords_denorm = invert_coordinate_normalisation(coords_roundtrip, normalization_stats)

print('coords_centered:', tuple(coords_centered.shape))
print('coords_norm:', tuple(coords_norm.shape))
print('coords_flat:', tuple(coords_flat.shape))
print('coords_roundtrip:', tuple(coords_roundtrip.shape))
print('coords_denorm:', tuple(coords_denorm.shape))
print('max inverse diff:', float((coords_denorm - coords_centered).abs().max().item()))
print('smoke-test mask sum:', float(mask.sum().item()))


## 6. Diffusion utilities and model

V4 keeps the V3b geometry-augmented DDPM objective but changes the denoiser architecture. The new model treats each `N`, `CA`, `C`, and `O` atom as a graph node with atom-type identity, residue-position conditioning, timestep conditioning, and sparse backbone edges. EGNN-style message passing uses squared distances for hidden-state updates and relative-coordinate updates for equivariant coordinate refinement.


In [ ]:
TIMESTEPS = 100
import math

# --- V4h FIX: cosine noise schedule with (near) zero terminal SNR ----------
# Root cause of the V4 collapse: the linear schedule run at only T=100 steps
# leaves alpha_bar_T = 0.364 (sqrt = 0.60), so ~60% of the clean structure is
# still present at the noisiest training step. But sampling starts from pure
# N(0, I) noise the model never trained on -> a built-in ~0.34x contraction
# that shrinks global Rg before the model gets a say (prior Rg ~20.3 A x 0.335
# = 6.8 A, matching the observed reverse-trajectory start of 6.46 A).
# A cosine schedule (Nichol & Dhariwal 2021) drives alpha_bar_T to ~2.4e-7
# (terminal SNR ~ 0) so pure-noise sampling finally matches training, WITHOUT
# the divide-by-zero that an exact-zero epsilon-prediction setup would hit.
def create_cosine_noise_schedule(timesteps, s=0.008, max_beta=0.999, device=None):
    """Cosine DDPM schedule, drop-in compatible with create_noise_schedule().

    Returns 1-indexed tensors (dummy entry at index 0) so timestep ``t`` indexes
    directly, matching the linear schedule's dict layout and downstream usage.
    """
    if timesteps <= 0:
        raise ValueError("timesteps must be positive.")
    steps = torch.arange(timesteps + 1, dtype=torch.float64)
    f = torch.cos(((steps / timesteps) + s) / (1 + s) * math.pi / 2) ** 2
    alpha_bars_full = f / f[0]  # alpha_bar at t = 0..T, with [0] == 1
    betas_1d = 1.0 - (alpha_bars_full[1:] / alpha_bars_full[:-1])
    betas_1d = betas_1d.clamp(1e-8, max_beta)  # clamp avoids end-of-schedule singularities
    alphas_1d = 1.0 - betas_1d
    # Recompute alpha_bars from the clamped betas so forward (q_sample) and
    # reverse (sampler) stay mutually consistent.
    alpha_bars_1d = torch.cumprod(alphas_1d, dim=0)
    alpha_bars_prev_1d = torch.cat([torch.ones(1, dtype=torch.float64), alpha_bars_1d[:-1]])
    posterior_variance_1d = betas_1d * (1.0 - alpha_bars_prev_1d) / (1.0 - alpha_bars_1d).clamp_min(1e-20)
    posterior_variance_1d[0] = 0.0

    zeros = torch.zeros(1, dtype=torch.float64)
    ones = torch.ones(1, dtype=torch.float64)
    schedule = {
        "betas": torch.cat([zeros, betas_1d]),
        "alphas": torch.cat([ones, alphas_1d]),
        "alpha_bars": torch.cat([ones, alpha_bars_1d]),
        "alpha_bars_prev": torch.cat([ones, alpha_bars_prev_1d]),
        "posterior_variance": torch.cat([zeros, posterior_variance_1d]),
    }
    return {key: value.to(dtype=torch.float32, device=device) for key, value in schedule.items()}

NOISE_SCHEDULE_TYPE = os.environ.get("BACKBONE_DIFFUSION_SCHEDULE", "cosine").strip().lower()
if NOISE_SCHEDULE_TYPE == "linear":
    noise_schedule = create_noise_schedule(TIMESTEPS, device=device)
else:
    noise_schedule = create_cosine_noise_schedule(TIMESTEPS, device=device)

# Terminal-SNR sanity check: alpha_bar_T must be ~0 so that pure-noise sampling
# matches the noisiest training step. This is the V4g bug this notebook fixes.
_linear_reference = create_noise_schedule(TIMESTEPS, device=device)
_alpha_bar_T = float(noise_schedule["alpha_bars"][-1])
_alpha_bar_T_linear = float(_linear_reference["alpha_bars"][-1])
print(f"Noise schedule type: {NOISE_SCHEDULE_TYPE}")
print(
    f"  alpha_bar_T (this run)   = {_alpha_bar_T:.3e}  sqrt = {_alpha_bar_T ** 0.5:.5f}  "
    f"terminal SNR = {_alpha_bar_T / (1 - _alpha_bar_T):.3e}"
)
print(
    f"  alpha_bar_T (linear ref) = {_alpha_bar_T_linear:.5f}  sqrt = {_alpha_bar_T_linear ** 0.5:.4f}  "
    f"terminal SNR = {_alpha_bar_T_linear / (1 - _alpha_bar_T_linear):.4f}"
)
assert _alpha_bar_T < 1e-3, (
    f"Terminal SNR is not ~0 (alpha_bar_T={_alpha_bar_T:.4f}). Pure-noise sampling will not match "
    "training and global collapse is expected. Use the cosine schedule or raise TIMESTEPS."
)
print("Terminal-SNR sanity check passed: pure-noise sampling now matches the noisiest training step.")

MODEL_TYPE = 'BackboneCoordinateEGNNDenoiser'
MODEL_PREDICTION_HEAD = 'hidden_state_plus_coordinate_residual'
MODEL_HIDDEN_DIM = int(os.environ.get('BACKBONE_DIFFUSION_MODEL_HIDDEN_DIM', '192'))
MODEL_NUM_LAYERS = int(os.environ.get('BACKBONE_DIFFUSION_MODEL_NUM_LAYERS', '4'))
MODEL_TIME_EMBEDDING_DIM = int(os.environ.get('BACKBONE_DIFFUSION_MODEL_TIME_EMBEDDING_DIM', '128'))
MODEL_SEQUENCE_OFFSET_EDGES = tuple(
    int(value) for value in os.environ.get('BACKBONE_DIFFUSION_SEQUENCE_OFFSET_EDGES', '8,16').split(',')
    if value.strip()
)
LEARNING_RATE = float(os.environ.get('BACKBONE_DIFFUSION_LEARNING_RATE', '1e-3'))
model = BackboneCoordinateEGNNDenoiser(
    max_length=MAX_SEQ_LENGTH,
    hidden_dim=MODEL_HIDDEN_DIM,
    num_layers=MODEL_NUM_LAYERS,
    time_embedding_dim=MODEL_TIME_EMBEDDING_DIM,
    sequence_offset_edges=MODEL_SEQUENCE_OFFSET_EDGES,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
print('Model type:', MODEL_TYPE)
print('Model prediction head:', MODEL_PREDICTION_HEAD)
print('Model parameters:', sum(parameter.numel() for parameter in model.parameters()))
print('Learning rate:', LEARNING_RATE)
print('Sequence-offset CA edges:', MODEL_SEQUENCE_OFFSET_EDGES)
print('Schedule keys:', list(noise_schedule.keys()))
print('Schedule shapes:')
for key, value in noise_schedule.items():
    print(key, tuple(value.shape), value.device)

with torch.no_grad():
    demo_coords = coords_norm.to(device)
    demo_mask = mask.to(device)
    demo_t = sample_timesteps(demo_coords.shape[0], TIMESTEPS, device)
    demo_noise = torch.randn_like(flatten_backbone(demo_coords))
    demo_x0 = flatten_backbone(demo_coords)
    demo_xt = q_sample(demo_x0, demo_t, demo_noise, noise_schedule['alpha_bars'])
    demo_pred = model(demo_xt, demo_t, demo_mask)
    demo_noise_loss = masked_noise_mse(demo_pred, demo_noise, demo_mask)

print('x0 shape:', tuple(demo_x0.shape))
print('xt shape:', tuple(demo_xt.shape))
print('pred shape:', tuple(demo_pred.shape))
print('demo masked noise loss:', float(demo_noise_loss.detach().cpu()))
if getattr(model, 'last_forward_stats', None):
    print('Demo forward stats:', model.last_forward_stats)
assert demo_pred.shape == demo_x0.shape



## 7. Training, validation, and test evaluation

V4h deliberately keeps the V4 training objective rather than adding a new nonlocal training loss. The objective remains:

```text
noise_mse
+ 0.01 * backbone_bond_geometry_loss(x0_pred)
+ 0.01 * adjacent_ca_geometry_loss(x0_pred)
```

Local geometry losses are active only for lower-noise timesteps (`t <= 50`). There is no radius loss and no nonlocal C-alpha loss during training. The new experiment happens later, during sampling, where the trained V4-style denoiser is evaluated with and without sampling-time guidance/projection.


In [ ]:
def prepare_batch_for_diffusion(batch: dict[str, object], target_device: torch.device, stats: BackboneNormalizationStats) -> tuple[torch.Tensor, torch.Tensor]:
    """Move a batch to device and return normalized flattened coords plus the mask."""
    batch = move_batch_to_device(batch, target_device)
    coords = batch['coords']
    mask = batch['mask']
    coords_centered = centre_coordinates(coords, mask)
    coords_norm = apply_coordinate_normalisation(coords_centered, mask, stats)
    coords_flat = flatten_backbone(coords_norm)
    return coords_flat, mask

NOISE_LOSS_WEIGHT = 1.0
LAMBDA_BOND = float(os.environ.get('BACKBONE_DIFFUSION_LAMBDA_BOND', '0.01'))
LAMBDA_CA = float(os.environ.get('BACKBONE_DIFFUSION_LAMBDA_CA', '0.01'))
GEOMETRY_LOSS_BETA = float(os.environ.get('BACKBONE_DIFFUSION_GEOMETRY_LOSS_BETA', '0.5'))
GEOMETRY_MAX_TIMESTEP = int(os.environ.get('BACKBONE_DIFFUSION_GEOMETRY_MAX_TIMESTEP', '50'))
GRAD_CLIP_NORM = float(os.environ.get('BACKBONE_DIFFUSION_GRAD_CLIP_NORM', '1.0'))
PRED_NEAR_ZERO_THRESHOLD = float(os.environ.get('BACKBONE_DIFFUSION_PRED_NEAR_ZERO_THRESHOLD', '1e-3'))
GEOMETRY_TARGETS_ANGSTROM = {
    'n_ca': 1.46,
    'ca_c': 1.53,
    'c_o': 1.23,
    'c_n': 1.33,
    'adjacent_ca': 3.80,
}

def geometry_timestep_mask(t: torch.Tensor, target_ndim: int) -> torch.Tensor:
    """Return a broadcastable mask that enables geometry loss only for selected timesteps."""
    active = (t <= GEOMETRY_MAX_TIMESTEP).float()
    view_shape = (t.shape[0],) + (1,) * (target_ndim - 1)
    return active.view(view_shape)

def masked_distance_huber(distances: torch.Tensor, target: float, valid_mask: torch.Tensor) -> torch.Tensor:
    """Smooth L1 distance error over valid bond or adjacent-residue positions."""
    valid = valid_mask.to(device=distances.device, dtype=distances.dtype)
    target_distances = torch.full_like(distances, fill_value=target)
    per_distance_loss = torch.nn.functional.smooth_l1_loss(
        distances,
        target_distances,
        beta=GEOMETRY_LOSS_BETA,
        reduction='none',
    )
    denom = valid.sum().clamp_min(1.0)
    return (per_distance_loss * valid).sum() / denom

def masked_tensor_rms(values: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Masked RMS over valid residues only."""
    mask_expanded = mask.unsqueeze(-1).expand_as(values).float()
    denom = mask_expanded.sum().clamp_min(1.0)
    return torch.sqrt(((values.pow(2) * mask_expanded).sum()) / denom)

def differentiable_predict_x0(
    x_t: torch.Tensor,
    t: torch.Tensor,
    pred_noise: torch.Tensor,
    alpha_bars: torch.Tensor,
) -> torch.Tensor:
    """Reconstruct x0 while preserving gradients for geometry losses."""
    alpha_bar_t = alpha_bars[t].view(-1, 1, 1).to(device=x_t.device, dtype=x_t.dtype)
    return (x_t - torch.sqrt(1.0 - alpha_bar_t) * pred_noise) / torch.sqrt(alpha_bar_t)

def x0_pred_to_angstrom_coords(x0_pred: torch.Tensor, stats: BackboneNormalizationStats) -> torch.Tensor:
    """Convert normalized flattened x0 prediction to B x L x 4 x 3 Angstrom coordinates."""
    return invert_coordinate_normalisation(unflatten_backbone(x0_pred), stats)

def adjacent_ca_geometry_loss(x0_pred: torch.Tensor, mask: torch.Tensor, t: torch.Tensor, stats: BackboneNormalizationStats) -> torch.Tensor:
    """Penalize adjacent CA distances at timesteps where x0 predictions are stable enough."""
    coords_angstrom = x0_pred_to_angstrom_coords(x0_pred, stats)
    ca = coords_angstrom[:, :, 1, :]
    adjacent_mask = mask[:, :-1].bool() & mask[:, 1:].bool()
    adjacent_mask = adjacent_mask.float() * geometry_timestep_mask(t, target_ndim=2).to(device=mask.device)
    adjacent_ca = torch.linalg.norm(ca[:, 1:, :] - ca[:, :-1, :], dim=-1)
    return masked_distance_huber(adjacent_ca, GEOMETRY_TARGETS_ANGSTROM['adjacent_ca'], adjacent_mask)

def backbone_bond_geometry_loss(x0_pred: torch.Tensor, mask: torch.Tensor, t: torch.Tensor, stats: BackboneNormalizationStats) -> torch.Tensor:
    """Penalize backbone bond lengths at timesteps where x0 predictions are stable enough."""
    coords_angstrom = x0_pred_to_angstrom_coords(x0_pred, stats)
    timestep_mask = geometry_timestep_mask(t, target_ndim=2).to(device=mask.device)
    residue_mask = mask.bool().float() * timestep_mask
    adjacent_mask = (mask[:, :-1].bool() & mask[:, 1:].bool()).float() * timestep_mask

    n_ca = torch.linalg.norm(coords_angstrom[:, :, 0, :] - coords_angstrom[:, :, 1, :], dim=-1)
    ca_c = torch.linalg.norm(coords_angstrom[:, :, 1, :] - coords_angstrom[:, :, 2, :], dim=-1)
    c_o = torch.linalg.norm(coords_angstrom[:, :, 2, :] - coords_angstrom[:, :, 3, :], dim=-1)
    c_n = torch.linalg.norm(coords_angstrom[:, :-1, 2, :] - coords_angstrom[:, 1:, 0, :], dim=-1)

    components = torch.stack([
        masked_distance_huber(n_ca, GEOMETRY_TARGETS_ANGSTROM['n_ca'], residue_mask),
        masked_distance_huber(ca_c, GEOMETRY_TARGETS_ANGSTROM['ca_c'], residue_mask),
        masked_distance_huber(c_o, GEOMETRY_TARGETS_ANGSTROM['c_o'], residue_mask),
        masked_distance_huber(c_n, GEOMETRY_TARGETS_ANGSTROM['c_n'], adjacent_mask),
    ])
    return components.mean()

def run_epoch(
    model: torch.nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer | None,
    stats: BackboneNormalizationStats,
    schedule: dict[str, torch.Tensor],
    target_device: torch.device,
) -> dict[str, float]:
    """Run one training or evaluation epoch."""
    is_training = optimizer is not None
    model.train(is_training)
    totals = defaultdict(float)
    n_batches = 0

    for batch in loader:
        coords_flat, mask = prepare_batch_for_diffusion(batch, target_device, stats)
        t = sample_timesteps(coords_flat.shape[0], TIMESTEPS, target_device)
        noise = torch.randn_like(coords_flat)
        x_t = q_sample(coords_flat, t, noise, schedule['alpha_bars'])

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            pred_noise = model(x_t, t, mask)
            debug_stats = getattr(model, 'last_forward_stats', {})
            noise_loss = masked_noise_mse(pred_noise, noise, mask)
            x0_pred = differentiable_predict_x0(x_t, t, pred_noise, schedule['alpha_bars'])
            bond_loss = backbone_bond_geometry_loss(x0_pred, mask, t, stats)
            ca_loss = adjacent_ca_geometry_loss(x0_pred, mask, t, stats)
            total_loss = (NOISE_LOSS_WEIGHT * noise_loss) + (LAMBDA_BOND * bond_loss) + (LAMBDA_CA * ca_loss)
            recon_rmse = masked_coordinate_rmse(x0_pred, coords_flat, mask)
            grad_norm = 0.0
            if is_training:
                total_loss.backward()
                grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM))
                optimizer.step()

        totals['total_loss'] += float(total_loss.detach().cpu())
        totals['noise_loss'] += float(noise_loss.detach().cpu())
        totals['bond_geometry_loss'] += float(bond_loss.detach().cpu())
        totals['adjacent_ca_geometry_loss'] += float(ca_loss.detach().cpu())
        totals['x0_rmse'] += float(recon_rmse.detach().cpu())
        totals['pred_noise_rms'] += float(masked_tensor_rms(pred_noise, mask).detach().cpu())
        totals['x_t_rms'] += float(masked_tensor_rms(x_t, mask).detach().cpu())
        totals['x0_pred_rms'] += float(masked_tensor_rms(x0_pred, mask).detach().cpu())
        totals['near_zero_fraction'] += float(debug_stats.get(
            'near_zero_fraction',
            (((pred_noise.abs() < PRED_NEAR_ZERO_THRESHOLD).float() * mask.unsqueeze(-1).float()).sum() / mask.unsqueeze(-1).expand_as(pred_noise).float().sum().clamp_min(1.0)).detach().cpu(),
        ))
        totals['coord_residual_rms'] += float(debug_stats.get('coord_residual_rms', 0.0))
        totals['node_head_rms'] += float(debug_stats.get('node_head_rms', 0.0))
        totals['coord_update_rms'] += float(debug_stats.get('coord_update_rms', 0.0))
        totals['grad_norm'] += grad_norm
        n_batches += 1

    return {key: value / max(n_batches, 1) for key, value in totals.items()}

EPOCHS = int(os.environ.get('BACKBONE_DIFFUSION_EPOCHS', '40'))
history: list[dict[str, float]] = []
best_val_total_loss = float('inf')

print(
    f'Geometry loss settings: bond={LAMBDA_BOND}, adjacent_ca={LAMBDA_CA}, '
    f'beta={GEOMETRY_LOSS_BETA}, active_t<= {GEOMETRY_MAX_TIMESTEP}, radius=0.0'
)
print(
    f'Optimizer settings: lr={LEARNING_RATE}, grad_clip_norm={GRAD_CLIP_NORM}, '
    f'sequence_offset_edges={MODEL_SEQUENCE_OFFSET_EDGES}'
)

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    train_metrics = run_epoch(model, train_loader, optimizer, normalization_stats, noise_schedule, device)
    val_metrics = run_epoch(model, validation_loader, None, normalization_stats, noise_schedule, device)
    test_metrics = run_epoch(model, test_loader, None, normalization_stats, noise_schedule, device)

    row = {
        'epoch': epoch,
        'train_total_loss': train_metrics['total_loss'],
        'train_noise_loss': train_metrics['noise_loss'],
        'train_bond_geometry_loss': train_metrics['bond_geometry_loss'],
        'train_adjacent_ca_geometry_loss': train_metrics['adjacent_ca_geometry_loss'],
        'train_x0_rmse': train_metrics['x0_rmse'],
        'train_pred_noise_rms': train_metrics['pred_noise_rms'],
        'train_x_t_rms': train_metrics['x_t_rms'],
        'train_x0_pred_rms': train_metrics['x0_pred_rms'],
        'train_near_zero_fraction': train_metrics['near_zero_fraction'],
        'train_coord_residual_rms': train_metrics['coord_residual_rms'],
        'train_node_head_rms': train_metrics['node_head_rms'],
        'train_coord_update_rms': train_metrics['coord_update_rms'],
        'train_grad_norm': train_metrics['grad_norm'],
        'validation_total_loss': val_metrics['total_loss'],
        'validation_noise_loss': val_metrics['noise_loss'],
        'validation_bond_geometry_loss': val_metrics['bond_geometry_loss'],
        'validation_adjacent_ca_geometry_loss': val_metrics['adjacent_ca_geometry_loss'],
        'validation_x0_rmse': val_metrics['x0_rmse'],
        'validation_pred_noise_rms': val_metrics['pred_noise_rms'],
        'validation_x_t_rms': val_metrics['x_t_rms'],
        'validation_x0_pred_rms': val_metrics['x0_pred_rms'],
        'validation_near_zero_fraction': val_metrics['near_zero_fraction'],
        'validation_coord_residual_rms': val_metrics['coord_residual_rms'],
        'validation_node_head_rms': val_metrics['node_head_rms'],
        'validation_coord_update_rms': val_metrics['coord_update_rms'],
        'test_total_loss': test_metrics['total_loss'],
        'test_noise_loss': test_metrics['noise_loss'],
        'test_bond_geometry_loss': test_metrics['bond_geometry_loss'],
        'test_adjacent_ca_geometry_loss': test_metrics['adjacent_ca_geometry_loss'],
        'test_x0_rmse': test_metrics['x0_rmse'],
        'test_pred_noise_rms': test_metrics['pred_noise_rms'],
        'test_x_t_rms': test_metrics['x_t_rms'],
        'test_x0_pred_rms': test_metrics['x0_pred_rms'],
        'test_near_zero_fraction': test_metrics['near_zero_fraction'],
        'test_coord_residual_rms': test_metrics['coord_residual_rms'],
        'test_node_head_rms': test_metrics['node_head_rms'],
        'test_coord_update_rms': test_metrics['coord_update_rms'],
        'lr': optimizer.param_groups[0]['lr'],
        'epoch_seconds': time.time() - start_time,
    }
    history.append(row)
    if row['validation_total_loss'] < best_val_total_loss:
        best_val_total_loss = row['validation_total_loss']
        torch.save(
            {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'normalization_mean': normalization_stats.mean,
                'normalization_std': normalization_stats.std,
                'timesteps': TIMESTEPS,
                'max_length': MAX_SEQ_LENGTH,
                'batch_size': BATCH_SIZE,
                'seed': SEED,
                'model_type': MODEL_TYPE,
                'model_prediction_head': MODEL_PREDICTION_HEAD,
                'learning_rate': LEARNING_RATE,
                'grad_clip_norm': GRAD_CLIP_NORM,
                'noise_loss_weight': NOISE_LOSS_WEIGHT,
                'lambda_bond': LAMBDA_BOND,
                'lambda_ca': LAMBDA_CA,
                'geometry_loss_beta': GEOMETRY_LOSS_BETA,
                'geometry_max_timestep': GEOMETRY_MAX_TIMESTEP,
                'geometry_loss_type': 'smooth_l1',
                'geometry_targets_angstrom': GEOMETRY_TARGETS_ANGSTROM,
                'pred_near_zero_threshold': PRED_NEAR_ZERO_THRESHOLD,
                'model_hidden_dim': MODEL_HIDDEN_DIM,
                'model_num_layers': MODEL_NUM_LAYERS,
                'model_time_embedding_dim': MODEL_TIME_EMBEDDING_DIM,
                'model_sequence_offset_edges': MODEL_SEQUENCE_OFFSET_EDGES,
                'saved_epoch': epoch,
                'saved_validation_total_loss': row['validation_total_loss'],
                'saved_validation_noise_loss': row['validation_noise_loss'],
            },
            CHECKPOINT_DIR / 'best_backbone_diffusion.pt',
        )
    print(
        f"Epoch {epoch:02d} | train total {row['train_total_loss']:.5f} | val total {row['validation_total_loss']:.5f} | "
        f"val noise {row['validation_noise_loss']:.5f} | val bond {row['validation_bond_geometry_loss']:.4f} | "
        f"val CA {row['validation_adjacent_ca_geometry_loss']:.4f} | train pred rms {row['train_pred_noise_rms']:.4f} | "
        f"val pred rms {row['validation_pred_noise_rms']:.4f} | val near-zero {row['validation_near_zero_fraction']:.3f} | "
        f"{row['epoch_seconds']:.1f}s"
    )

history_df = pd.DataFrame(history)
display(history_df)
save_table_artifact(history_df, 'backbone_diffusion_history')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(history_df['epoch'], history_df['train_total_loss'], label='Train total')
axes[0].plot(history_df['epoch'], history_df['validation_total_loss'], label='Validation total')
axes[0].plot(history_df['epoch'], history_df['test_total_loss'], label='Test total')
axes[0].plot(history_df['epoch'], history_df['validation_noise_loss'], linestyle='--', label='Validation noise')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('V4h training objective')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['validation_bond_geometry_loss'], marker='o', label='Validation bond geometry')
axes[1].plot(history_df['epoch'], history_df['validation_adjacent_ca_geometry_loss'], marker='o', label='Validation adjacent CA')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Unweighted Smooth L1 geometry loss')
axes[1].set_title('V4h geometry losses')
axes[1].legend()

axes[2].plot(history_df['epoch'], history_df['train_pred_noise_rms'], marker='o', label='Train pred noise RMS')
axes[2].plot(history_df['epoch'], history_df['validation_pred_noise_rms'], marker='o', label='Validation pred noise RMS')
axes[2].plot(history_df['epoch'], history_df['validation_near_zero_fraction'], marker='s', linestyle='--', label='Validation near-zero frac')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('RMS / fraction')
axes[2].set_title('V4h output diagnostics')
axes[2].legend()

fig.tight_layout()
fig.savefig(FIGURE_DIR / 'backbone_diffusion_losses.png', dpi=150)
plt.show()


## 8. Sampling and evaluation

This section reloads the saved best-validation checkpoint before sampling, then denormalizes sampled backbones back into coordinate space for structural evaluation and visualization. That keeps the reported qualitative and quantitative outputs aligned with the model-selection rule used during training.


In [ ]:
checkpoint_path = CHECKPOINT_DIR / 'best_backbone_diffusion.pt'
checkpoint = torch.load(checkpoint_path, map_location=device)
try:
    model.load_state_dict(checkpoint['model_state_dict'])
except RuntimeError as exc:
    raise RuntimeError(
        'The saved V4h/V4-style checkpoint is incompatible with the updated troubleshooting architecture. '
        'Rerun section 7 training to write a fresh results/v4h/checkpoints/best_backbone_diffusion.pt checkpoint.'
    ) from exc
print(
    f"Loaded best checkpoint from {checkpoint_path} | epoch={checkpoint.get('saved_epoch', 'unknown')} | "
    f"val_total={checkpoint.get('saved_validation_total_loss', 'unknown')} | "
    f"val_noise={checkpoint.get('saved_validation_noise_loss', 'unknown')}"
)

model.eval()
with torch.no_grad():
    eval_batch = next(iter(validation_loader))
    sample_mask = eval_batch['mask'][:4].to(device)
    sampled_flat = sample_backbone(
        model,
        noise_schedule,
        shape=(sample_mask.shape[0], MAX_SEQ_LENGTH, 12),
        mask=sample_mask,
        device=device,
    )
    sampled_coords_norm = unflatten_backbone(sampled_flat)
    sampled_coords = invert_coordinate_normalisation(sampled_coords_norm, normalization_stats)

    real_coords = eval_batch['coords'][:4]
    real_mask = eval_batch['mask'][:4]

real_rows = []
generated_rows = []
for index in range(sampled_coords.shape[0]):
    real_rows.append({'kind': 'real', 'index': index, **backbone_structure_summary(real_coords[index], real_mask[index])})
    generated_rows.append({'kind': 'generated', 'index': index, **backbone_structure_summary(sampled_coords[index].cpu(), sample_mask[index].cpu())})

real_eval_df = pd.DataFrame(real_rows)
generated_eval_df = pd.DataFrame(generated_rows)

display(real_eval_df)
display(generated_eval_df)

comparison_df = pd.concat([real_eval_df, generated_eval_df], ignore_index=True)
comparison_summary = comparison_df.groupby('kind', as_index=False).mean(numeric_only=True)
display(comparison_summary)
save_table_artifact(real_eval_df, 'real_eval_metrics')
save_table_artifact(generated_eval_df, 'generated_eval_metrics')
save_table_artifact(comparison_df, 'real_vs_generated_summary')
save_table_artifact(comparison_summary, 'real_vs_generated_summary_mean')


In [ ]:
for index in range(min(2, sampled_coords.shape[0])):
    print(f'Generated example {index}')
    print(backbone_structure_summary(sampled_coords[index].cpu(), sample_mask[index].cpu()))
    plot_ca_trace_from_batch(sampled_coords[index].cpu(), sample_mask[index].bool().cpu(), title=f'Generated CA trace {index}')
    render_backbone_example(sampled_coords[index].cpu(), sample_mask[index].cpu(), title=f'Generated backbone {index}')



## 9. V4h diagnostic evaluation

V4h keeps the V4 diagnostic harness and adds a sampling-time guidance comparison. The baseline sampler is the usual DDPM reverse process. The guided samplers add small projection steps during reverse diffusion:

- a length-aware radius projection, applied mainly in the high/mid-noise part of sampling, to discourage globally over-compact chains.
- an optional adjacent C-alpha residue-translation projection, applied mainly in the lower-noise part of sampling, to preserve local trace continuity after global expansion.

This is intentionally a heuristic prototype, not a replacement for a fully conditioned protein diffusion model. The point is to test whether the remaining collapse problem is better addressed at sampling time than by adding more training-time nonlocal losses.


In [ ]:
model_parameter_count = sum(parameter.numel() for parameter in model.parameters())
train_chunk_count_summary = train_chunked_df.groupby('parent_chain_id').size()
best_epoch_row = history_df.loc[history_df['validation_total_loss'].idxmin()].to_dict()

def config_row(key: str, value: object) -> dict[str, object]:
    """Return a Parquet-safe config row with stable column types."""
    is_bool = isinstance(value, bool)
    is_int = isinstance(value, int) and not is_bool
    is_float = isinstance(value, float)
    return {
        'key': key,
        'value_text': str(value),
        'value_int': int(value) if is_int else pd.NA,
        'value_float': float(value) if (is_int or is_float) else np.nan,
    }

run_config_summary = pd.DataFrame([
    config_row('artifact_run_name', ARTIFACT_RUN_NAME),
    config_row('seed', SEED),
    config_row('device', str(device)),
    config_row('device_name', device_name),
    config_row('max_seq_length', MAX_SEQ_LENGTH),
    config_row('backbone_atoms', ','.join(BACKBONE_ATOMS)),
    config_row('train_chains', len(train_df)),
    config_row('validation_chains', len(validation_df)),
    config_row('test_chains', len(test_df)),
    config_row('train_chunks', len(train_chunked_df)),
    config_row('train_chunk_stride', TRAIN_CHUNK_STRIDE),
    config_row('median_chunks_per_train_chain', float(train_chunk_count_summary.median())),
    config_row('max_chunks_per_train_chain', int(train_chunk_count_summary.max())),
    config_row('overfit_debug', OVERFIT_DEBUG),
    config_row('overfit_sample_count', OVERFIT_SAMPLE_COUNT),
    config_row('overfit_batch_size', OVERFIT_BATCH_SIZE),
    config_row('overfit_use_train_subset_for_eval', OVERFIT_USE_TRAIN_SUBSET_FOR_EVAL),
    config_row('overfit_shuffle_train', OVERFIT_SHUFFLE_TRAIN),
    config_row('train_dataset_size_used', len(train_dataset)),
    config_row('validation_dataset_size_used', len(validation_dataset)),
    config_row('test_dataset_size_used', len(test_dataset)),
    config_row('timesteps', TIMESTEPS),
    config_row('epochs', EPOCHS),
    config_row('batch_size', BATCH_SIZE),
    config_row('optimizer', 'Adam'),
    config_row('learning_rate', LEARNING_RATE),
    config_row('grad_clip_norm', GRAD_CLIP_NORM),
    config_row('model_type', MODEL_TYPE),
    config_row('model_prediction_head', MODEL_PREDICTION_HEAD),
    config_row('model_hidden_dim', MODEL_HIDDEN_DIM),
    config_row('model_num_layers', MODEL_NUM_LAYERS),
    config_row('model_time_embedding_dim', MODEL_TIME_EMBEDDING_DIM),
    config_row('model_sequence_offset_edges', ','.join(str(value) for value in MODEL_SEQUENCE_OFFSET_EDGES)),
    config_row('model_parameter_count', model_parameter_count),
    config_row('noise_loss_weight', NOISE_LOSS_WEIGHT),
    config_row('lambda_bond', LAMBDA_BOND),
    config_row('lambda_ca', LAMBDA_CA),
    config_row('lambda_radius_of_gyration', 0.0),
    config_row('geometry_loss_type', 'smooth_l1'),
    config_row('geometry_loss_beta', GEOMETRY_LOSS_BETA),
    config_row('geometry_max_timestep', GEOMETRY_MAX_TIMESTEP),
    config_row('pred_near_zero_threshold', PRED_NEAR_ZERO_THRESHOLD),
    config_row('target_n_ca_angstrom', GEOMETRY_TARGETS_ANGSTROM['n_ca']),
    config_row('target_ca_c_angstrom', GEOMETRY_TARGETS_ANGSTROM['ca_c']),
    config_row('target_c_o_angstrom', GEOMETRY_TARGETS_ANGSTROM['c_o']),
    config_row('target_c_n_angstrom', GEOMETRY_TARGETS_ANGSTROM['c_n']),
    config_row('target_adjacent_ca_angstrom', GEOMETRY_TARGETS_ANGSTROM['adjacent_ca']),
    config_row('best_epoch', int(best_epoch_row['epoch'])),
    config_row('best_validation_total_loss', float(best_epoch_row['validation_total_loss'])),
    config_row('best_validation_noise_loss', float(best_epoch_row['validation_noise_loss'])),
    config_row('best_test_total_loss_at_reported_epoch', float(best_epoch_row['test_total_loss'])),
    config_row('best_test_noise_loss_at_reported_epoch', float(best_epoch_row['test_noise_loss'])),
])
run_config_summary['value_int'] = run_config_summary['value_int'].astype('Int64')

display(run_config_summary)
save_table_artifact(run_config_summary, 'v4h_run_config_summary')


### Fixed-timestep denoising diagnostics


In [ ]:
@torch.no_grad()
def evaluate_fixed_timesteps(
    model: torch.nn.Module,
    loader: DataLoader,
    split_name: str,
    timesteps_to_check: list[int],
    stats: BackboneNormalizationStats,
    schedule: dict[str, torch.Tensor],
    target_device: torch.device,
    max_batches: int | None = None,
) -> pd.DataFrame:
    model.eval()
    rows = []
    for timestep in timesteps_to_check:
        totals = defaultdict(float)
        n_batches = 0
        for batch_index, batch in enumerate(loader):
            if max_batches is not None and batch_index >= max_batches:
                break
            coords_flat, mask = prepare_batch_for_diffusion(batch, target_device, stats)
            t = torch.full((coords_flat.shape[0],), timestep, device=target_device, dtype=torch.long)
            noise = torch.randn_like(coords_flat)
            x_t = q_sample(coords_flat, t, noise, schedule['alpha_bars'])
            pred_noise = model(x_t, t, mask)
            debug_stats = getattr(model, 'last_forward_stats', {})
            noise_loss = masked_noise_mse(pred_noise, noise, mask)
            x0_pred = differentiable_predict_x0(x_t, t, pred_noise, schedule['alpha_bars'])
            bond_loss = backbone_bond_geometry_loss(x0_pred, mask, t, stats)
            ca_loss = adjacent_ca_geometry_loss(x0_pred, mask, t, stats)
            total_loss = (NOISE_LOSS_WEIGHT * noise_loss) + (LAMBDA_BOND * bond_loss) + (LAMBDA_CA * ca_loss)
            recon_rmse = masked_coordinate_rmse(x0_pred, coords_flat, mask)
            totals['total_loss'] += float(total_loss.detach().cpu())
            totals['noise_loss'] += float(noise_loss.detach().cpu())
            totals['bond_geometry_loss'] += float(bond_loss.detach().cpu())
            totals['adjacent_ca_geometry_loss'] += float(ca_loss.detach().cpu())
            totals['x0_rmse'] += float(recon_rmse.detach().cpu())
            totals['pred_noise_rms'] += float(masked_tensor_rms(pred_noise, mask).detach().cpu())
            totals['x_t_rms'] += float(masked_tensor_rms(x_t, mask).detach().cpu())
            totals['x0_pred_rms'] += float(masked_tensor_rms(x0_pred, mask).detach().cpu())
            totals['near_zero_fraction'] += float(debug_stats.get('near_zero_fraction', 0.0))
            totals['coord_residual_rms'] += float(debug_stats.get('coord_residual_rms', 0.0))
            totals['node_head_rms'] += float(debug_stats.get('node_head_rms', 0.0))
            totals['coord_update_rms'] += float(debug_stats.get('coord_update_rms', 0.0))
            n_batches += 1
        rows.append({
            'split': split_name,
            'timestep': timestep,
            'noise_fraction': timestep / TIMESTEPS,
            'total_loss': totals['total_loss'] / max(n_batches, 1),
            'noise_loss': totals['noise_loss'] / max(n_batches, 1),
            'bond_geometry_loss': totals['bond_geometry_loss'] / max(n_batches, 1),
            'adjacent_ca_geometry_loss': totals['adjacent_ca_geometry_loss'] / max(n_batches, 1),
            'x0_rmse': totals['x0_rmse'] / max(n_batches, 1),
            'pred_noise_rms': totals['pred_noise_rms'] / max(n_batches, 1),
            'x_t_rms': totals['x_t_rms'] / max(n_batches, 1),
            'x0_pred_rms': totals['x0_pred_rms'] / max(n_batches, 1),
            'near_zero_fraction': totals['near_zero_fraction'] / max(n_batches, 1),
            'coord_residual_rms': totals['coord_residual_rms'] / max(n_batches, 1),
            'node_head_rms': totals['node_head_rms'] / max(n_batches, 1),
            'coord_update_rms': totals['coord_update_rms'] / max(n_batches, 1),
            'n_batches': n_batches,
        })
    return pd.DataFrame(rows)

TIMESTEP_DIAGNOSTIC_POINTS = [1, 5, 10, 25, 50, 75, 100]
validation_timestep_df = evaluate_fixed_timesteps(
    model,
    validation_loader,
    'validation',
    TIMESTEP_DIAGNOSTIC_POINTS,
    normalization_stats,
    noise_schedule,
    device,
)
test_timestep_df = evaluate_fixed_timesteps(
    model,
    test_loader,
    'test',
    TIMESTEP_DIAGNOSTIC_POINTS,
    normalization_stats,
    noise_schedule,
    device,
)
timestep_diagnostics_df = pd.concat([validation_timestep_df, test_timestep_df], ignore_index=True)

display(timestep_diagnostics_df)
save_table_artifact(timestep_diagnostics_df, 'v4h_timestep_diagnostics')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for split_name, split_df in timestep_diagnostics_df.groupby('split'):
    axes[0].plot(split_df['timestep'], split_df['noise_loss'], marker='o', label=f'{split_name} noise')
    axes[1].plot(split_df['timestep'], split_df['bond_geometry_loss'], marker='o', label=f'{split_name} bond')
    axes[1].plot(split_df['timestep'], split_df['adjacent_ca_geometry_loss'], marker='s', linestyle='--', label=f'{split_name} CA')
    axes[2].plot(split_df['timestep'], split_df['pred_noise_rms'], marker='o', label=f'{split_name} pred rms')
    axes[2].plot(split_df['timestep'], split_df['near_zero_fraction'], marker='s', linestyle='--', label=f'{split_name} near-zero')
axes[0].set_xlabel('Diffusion timestep')
axes[0].set_ylabel('Masked noise MSE')
axes[0].set_title('V4h fixed-timestep denoising')
axes[0].legend()
axes[1].set_xlabel('Diffusion timestep')
axes[1].set_ylabel('Smooth L1 geometry loss')
axes[1].set_title('V4h fixed-timestep geometry')
axes[1].legend()
axes[2].set_xlabel('Diffusion timestep')
axes[2].set_ylabel('RMS / fraction')
axes[2].set_title('V4h fixed-timestep output diagnostics')
axes[2].legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'v4h_timestep_diagnostics.png', dpi=150)
plt.show()


### Broader sample-quality diagnostics

V4h samples the same number of validation-shaped masks as V2/V3/V3b and computes the same structural summaries. When reference artifact tables are present, the notebook writes direct metric and collapse comparisons against V4 and V3b, then adds explicit nonlocal C-alpha summaries to check whether the generated structures are globally less over-compact than V4.


In [ ]:
@torch.no_grad()
def collect_conditioning_masks(loader: DataLoader, target_count: int) -> tuple[torch.Tensor, torch.Tensor]:
    coords_batches = []
    mask_batches = []
    seen = 0
    for batch in loader:
        coords = batch['coords']
        mask = batch['mask']
        remaining = target_count - seen
        coords_batches.append(coords[:remaining])
        mask_batches.append(mask[:remaining])
        seen += min(coords.shape[0], remaining)
        if seen >= target_count:
            break
    return torch.cat(coords_batches, dim=0), torch.cat(mask_batches, dim=0)

@torch.no_grad()
def sample_and_evaluate_masks(
    model: torch.nn.Module,
    masks: torch.Tensor,
    real_coords: torch.Tensor,
    stats: BackboneNormalizationStats,
    schedule: dict[str, torch.Tensor],
    target_device: torch.device,
    sample_batch_size: int = 8,
) -> tuple[pd.DataFrame, pd.DataFrame, torch.Tensor]:
    model.eval()
    generated_coord_batches = []
    real_rows = []
    generated_rows = []
    for start in range(0, masks.shape[0], sample_batch_size):
        end = min(start + sample_batch_size, masks.shape[0])
        batch_mask = masks[start:end].to(target_device)
        sampled_flat = sample_backbone(
            model,
            schedule,
            shape=(batch_mask.shape[0], MAX_SEQ_LENGTH, 12),
            mask=batch_mask,
            device=target_device,
        )
        sampled_coords_norm = unflatten_backbone(sampled_flat)
        generated_coords = invert_coordinate_normalisation(sampled_coords_norm, stats).cpu()
        generated_coord_batches.append(generated_coords)
        for local_index in range(generated_coords.shape[0]):
            index = start + local_index
            real_rows.append({
                'kind': 'real',
                'index': index,
                **backbone_structure_summary(real_coords[index], masks[index]),
                **nonlocal_ca_contact_diagnostics(real_coords[index], masks[index]),
            })
            generated_rows.append({
                'kind': 'generated',
                'index': index,
                **backbone_structure_summary(generated_coords[local_index], masks[index]),
                **nonlocal_ca_contact_diagnostics(generated_coords[local_index], masks[index]),
            })
    return pd.DataFrame(real_rows), pd.DataFrame(generated_rows), torch.cat(generated_coord_batches, dim=0)

def build_metric_summary(comparison_df: pd.DataFrame) -> pd.DataFrame:
    return comparison_df.groupby('kind', as_index=False).agg(
        n_structures=('index', 'count'),
        mean_n_residues=('n_residues', 'mean'),
        mean_adjacent_ca=('mean_adjacent_ca', 'mean'),
        std_adjacent_ca=('mean_adjacent_ca', 'std'),
        mean_fraction_adjacent_ca_in_band=('fraction_adjacent_ca_in_band', 'mean'),
        mean_n_ca=('mean_n_ca', 'mean'),
        mean_ca_c=('mean_ca_c', 'mean'),
        mean_c_o=('mean_c_o', 'mean'),
        mean_c_n=('mean_c_n', 'mean'),
        mean_radius_of_gyration=('radius_of_gyration', 'mean'),
        std_radius_of_gyration=('radius_of_gyration', 'std'),
    )

def build_structural_gap_summary(metric_summary: pd.DataFrame) -> pd.DataFrame:
    real_means = metric_summary[metric_summary['kind'] == 'real'].iloc[0]
    generated_means = metric_summary[metric_summary['kind'] == 'generated'].iloc[0]
    return pd.DataFrame([
        {
            'metric': 'mean_adjacent_ca',
            'real_mean': real_means['mean_adjacent_ca'],
            'generated_mean': generated_means['mean_adjacent_ca'],
            'generated_minus_real': generated_means['mean_adjacent_ca'] - real_means['mean_adjacent_ca'],
            'interpretation': 'Generated adjacent CA spacing should be close to the real mean around 3.8 A.',
        },
        {
            'metric': 'fraction_adjacent_ca_in_band',
            'real_mean': real_means['mean_fraction_adjacent_ca_in_band'],
            'generated_mean': generated_means['mean_fraction_adjacent_ca_in_band'],
            'generated_minus_real': generated_means['mean_fraction_adjacent_ca_in_band'] - real_means['mean_fraction_adjacent_ca_in_band'],
            'interpretation': 'Low generated fraction indicates broken local CA geometry.',
        },
        {
            'metric': 'radius_of_gyration',
            'real_mean': real_means['mean_radius_of_gyration'],
            'generated_mean': generated_means['mean_radius_of_gyration'],
            'generated_minus_real': generated_means['mean_radius_of_gyration'] - real_means['mean_radius_of_gyration'],
            'interpretation': 'Low generated radius indicates global collapse or over-compact sampling.',
        },
    ])

def adjacent_ca_distribution_summary(eval_df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Summarize per-structure adjacent CA metrics beyond the mean."""
    frame = eval_df.copy()
    return pd.DataFrame([
        {
            'kind': label,
            'n_structures': len(frame),
            'mean_adjacent_ca_mean': frame['mean_adjacent_ca'].mean(),
            'mean_adjacent_ca_std': frame['mean_adjacent_ca'].std(),
            'mean_adjacent_ca_p05': frame['mean_adjacent_ca'].quantile(0.05),
            'mean_adjacent_ca_p25': frame['mean_adjacent_ca'].quantile(0.25),
            'mean_adjacent_ca_median': frame['mean_adjacent_ca'].median(),
            'mean_adjacent_ca_p75': frame['mean_adjacent_ca'].quantile(0.75),
            'mean_adjacent_ca_p95': frame['mean_adjacent_ca'].quantile(0.95),
            'ca_in_band_mean': frame['fraction_adjacent_ca_in_band'].mean(),
            'ca_in_band_p25': frame['fraction_adjacent_ca_in_band'].quantile(0.25),
            'ca_in_band_median': frame['fraction_adjacent_ca_in_band'].median(),
            'ca_in_band_p75': frame['fraction_adjacent_ca_in_band'].quantile(0.75),
        }
    ])

def nonlocal_ca_contact_diagnostics(
    coords: torch.Tensor,
    mask: torch.Tensor,
    min_sequence_separation: int = 8,
    thresholds: tuple[float, ...] = (8.0, 10.0),
) -> dict[str, float]:
    """Return nonlocal CA distance/contact diagnostics for one structure."""

    if coords.ndim == 4:
        coords = coords[0]
    if coords.ndim != 3 or coords.shape[-2:] != (4, 3):
        raise ValueError('coords must have shape (L, 4, 3).')
    if mask.ndim != 1:
        raise ValueError('mask must have shape (L,).')

    mask_bool = mask.bool()
    ca = coords[:, 1, :]
    valid_pair = mask_bool[:, None] & mask_bool[None, :]
    residue_index = torch.arange(coords.shape[0], device=coords.device)
    sequence_separation = residue_index[None, :] - residue_index[:, None]
    nonlocal_pair = sequence_separation >= min_sequence_separation
    pair_mask = valid_pair & nonlocal_pair

    if pair_mask.sum() == 0:
        return {
            'mean_nonlocal_ca_distance': float('nan'),
            'median_nonlocal_ca_distance': float('nan'),
            'fraction_nonlocal_ca_below_8A': 0.0,
            'fraction_nonlocal_ca_below_10A': 0.0,
            'max_pairwise_ca_distance': 0.0,
            'end_to_end_ca_distance': 0.0,
        }

    dist = torch.linalg.norm(ca[:, None, :] - ca[None, :, :], dim=-1)
    valid_distances = dist[pair_mask]
    end_to_end = torch.linalg.norm(ca[mask_bool][-1] - ca[mask_bool][0], dim=-1) if mask_bool.sum() > 1 else torch.tensor(0.0, device=coords.device, dtype=coords.dtype)
    return {
        'mean_nonlocal_ca_distance': float(valid_distances.mean().item()),
        'median_nonlocal_ca_distance': float(valid_distances.median().item()),
        'fraction_nonlocal_ca_below_8A': float((valid_distances < thresholds[0]).float().mean().item()),
        'fraction_nonlocal_ca_below_10A': float((valid_distances < thresholds[1]).float().mean().item()),
        'max_pairwise_ca_distance': float(valid_distances.max().item()),
        'end_to_end_ca_distance': float(end_to_end.item()),
    }


def load_reference_metric_summary(version: str) -> pd.DataFrame | None:
    reference_path = ARTIFACT_BASE_DIR / version / 'tables' / f'{version}_real_vs_generated_metric_summary.csv'
    if not reference_path.exists():
        print(f'{version.upper()} reference metric summary not found at {reference_path}; skipping direct metric comparison.')
        return None
    return pd.read_csv(reference_path)

def load_reference_collapse_summary(version: str) -> pd.DataFrame | None:
    reference_path = ARTIFACT_BASE_DIR / version / 'tables' / f'{version}_collapse_summary.csv'
    if not reference_path.exists():
        print(f'{version.upper()} reference collapse summary not found at {reference_path}; skipping collapse comparison.')
        return None
    return pd.read_csv(reference_path)

def build_metric_comparison(reference_version: str, reference_summary: pd.DataFrame, current_summary: pd.DataFrame, current_version: str) -> pd.DataFrame:
    reference_generated = reference_summary[reference_summary['kind'] == 'generated'].iloc[0]
    current_generated = current_summary[current_summary['kind'] == 'generated'].iloc[0]
    current_real = current_summary[current_summary['kind'] == 'real'].iloc[0]
    metrics = [
        ('mean_adjacent_ca', 'increase toward 3.8 A'),
        ('mean_fraction_adjacent_ca_in_band', 'increase'),
        ('mean_radius_of_gyration', 'increase toward real'),
        ('mean_n_ca', 'move toward real'),
        ('mean_ca_c', 'move toward real'),
        ('mean_c_o', 'move toward real'),
        ('mean_c_n', 'move toward real'),
    ]
    rows = []
    for metric, target_direction in metrics:
        rows.append({
            'metric': metric,
            'real_mean': current_real[metric],
            f'{reference_version}_generated_mean': reference_generated[metric],
            f'{current_version}_generated_mean': current_generated[metric],
            f'{current_version}_minus_{reference_version}': current_generated[metric] - reference_generated[metric],
            'target_direction': target_direction,
        })
    return pd.DataFrame(rows)

def build_collapse_comparison(reference_version: str, reference_collapse: pd.DataFrame, current_collapse: pd.DataFrame, current_version: str) -> pd.DataFrame:
    reference = reference_collapse.iloc[0]
    current = current_collapse.iloc[0]
    reference_sample_count = int(reference['sample_count'])
    current_sample_count = int(current['sample_count'])
    reference_collapse_count = int(round(float(reference['collapse_fraction']) * reference_sample_count))
    reference_poor_ca_count = int(round(float(reference['poor_ca_band_fraction']) * reference_sample_count))
    current_collapse_count = int(current.get('collapse_count', round(float(current['collapse_fraction']) * current_sample_count)))
    current_poor_ca_count = int(current.get('poor_ca_band_count', round(float(current['poor_ca_band_fraction']) * current_sample_count)))
    return pd.DataFrame([
        {
            'version': reference_version,
            'sample_count': reference_sample_count,
            'collapse_fraction': float(reference['collapse_fraction']),
            'collapse_count': reference_collapse_count,
            'poor_ca_band_fraction': float(reference['poor_ca_band_fraction']),
            'poor_ca_band_count': reference_poor_ca_count,
        },
        {
            'version': current_version,
            'sample_count': current_sample_count,
            'collapse_fraction': float(current['collapse_fraction']),
            'collapse_count': current_collapse_count,
            'poor_ca_band_fraction': float(current['poor_ca_band_fraction']),
            'poor_ca_band_count': current_poor_ca_count,
        },
    ])

V4H_SAMPLE_COUNT = min(32, len(validation_dataset))
v4h_real_coords, v4h_sample_masks = collect_conditioning_masks(validation_loader, V4H_SAMPLE_COUNT)
v4h_real_eval_df, v4h_generated_eval_df, v4h_sampled_coords = sample_and_evaluate_masks(
    model,
    v4h_sample_masks,
    v4h_real_coords,
    normalization_stats,
    noise_schedule,
    device,
    sample_batch_size=8,
)

v4h_comparison_df = pd.concat([v4h_real_eval_df, v4h_generated_eval_df], ignore_index=True)
v4h_metric_summary = build_metric_summary(v4h_comparison_df)
v4h_gap_summary = build_structural_gap_summary(v4h_metric_summary)

real_rg_median = float(v4h_real_eval_df['radius_of_gyration'].median())
collapse_radius_threshold = 0.5 * real_rg_median
v4h_generated_eval_df['collapse_flag'] = v4h_generated_eval_df['radius_of_gyration'] < collapse_radius_threshold
v4h_generated_eval_df['poor_ca_band_flag'] = v4h_generated_eval_df['fraction_adjacent_ca_in_band'] < 0.8
collapse_summary = pd.DataFrame([
    {
        'sample_count': len(v4h_generated_eval_df),
        'real_radius_median': real_rg_median,
        'collapse_radius_threshold': collapse_radius_threshold,
        'collapse_fraction': float(v4h_generated_eval_df['collapse_flag'].mean()),
        'collapse_count': int(v4h_generated_eval_df['collapse_flag'].sum()),
        'poor_ca_band_fraction': float(v4h_generated_eval_df['poor_ca_band_flag'].mean()),
        'poor_ca_band_count': int(v4h_generated_eval_df['poor_ca_band_flag'].sum()),
    }
])

for frame in [v4h_metric_summary, v4h_gap_summary, collapse_summary]:
    display(frame)

save_table_artifact(v4h_real_eval_df, 'v4h_real_eval_metrics')
save_table_artifact(v4h_generated_eval_df, 'v4h_generated_eval_metrics')
save_table_artifact(v4h_comparison_df, 'v4h_real_vs_generated_summary')
save_table_artifact(v4h_metric_summary, 'v4h_real_vs_generated_metric_summary')
save_table_artifact(v4h_gap_summary, 'v4h_structural_gap_summary')
save_table_artifact(collapse_summary, 'v4h_collapse_summary')

v4h_ca_distribution_summary = pd.concat([
    adjacent_ca_distribution_summary(v4h_real_eval_df, 'real'),
    adjacent_ca_distribution_summary(v4h_generated_eval_df, 'generated'),
], ignore_index=True)
display(v4h_ca_distribution_summary)
save_table_artifact(v4h_ca_distribution_summary, 'v4h_adjacent_ca_distribution_summary')

for reference_version in ['v4', 'v3b']:
    reference_metric_summary = load_reference_metric_summary(reference_version)
    if reference_metric_summary is not None:
        metric_comparison = build_metric_comparison(reference_version, reference_metric_summary, v4h_metric_summary, ARTIFACT_RUN_NAME)
        display(metric_comparison)
        save_table_artifact(metric_comparison, f'v4h_vs_{reference_version}_metric_comparison')

    reference_collapse_summary = load_reference_collapse_summary(reference_version)
    if reference_collapse_summary is not None:
        collapse_comparison = build_collapse_comparison(reference_version, reference_collapse_summary, collapse_summary, ARTIFACT_RUN_NAME)
        display(collapse_comparison)
        save_table_artifact(collapse_comparison, f'v4h_vs_{reference_version}_collapse_comparison')




In [ ]:

# V4h sampling-time guidance / projection comparison.
# This cell keeps the trained V4-style denoiser fixed and changes only the sampling procedure.

V4H_GUIDANCE_SAMPLE_COUNT = int(os.environ.get('BACKBONE_DIFFUSION_V4H_GUIDANCE_SAMPLE_COUNT', str(min(32, len(validation_dataset)))))
V4H_GUIDANCE_BATCH_SIZE = int(os.environ.get('BACKBONE_DIFFUSION_V4H_GUIDANCE_BATCH_SIZE', '8'))
V4H_RADIUS_GUIDANCE_START_TIMESTEP = int(os.environ.get('BACKBONE_DIFFUSION_V4H_RADIUS_GUIDANCE_START_TIMESTEP', '75'))
V4H_RADIUS_GUIDANCE_END_TIMESTEP = int(os.environ.get('BACKBONE_DIFFUSION_V4H_RADIUS_GUIDANCE_END_TIMESTEP', '10'))
V4H_LOCAL_PROJECTION_MAX_TIMESTEP = int(os.environ.get('BACKBONE_DIFFUSION_V4H_LOCAL_PROJECTION_MAX_TIMESTEP', '50'))


def _mask_flattened_coords(x_flat: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    return x_flat * mask.to(device=x_flat.device, dtype=x_flat.dtype).unsqueeze(-1)


def normalized_flat_to_centered_angstrom(x_flat: torch.Tensor, mask: torch.Tensor, stats: BackboneNormalizationStats) -> torch.Tensor:
    """Convert normalized flattened coordinates to centered Angstrom coordinates."""
    coords = invert_coordinate_normalisation(unflatten_backbone(x_flat), stats)
    return centre_coordinates(coords, mask)


def centered_angstrom_to_normalized_flat(coords: torch.Tensor, mask: torch.Tensor, stats: BackboneNormalizationStats) -> torch.Tensor:
    """Convert centered Angstrom coordinates back to normalized flattened coordinates."""
    coords_centered = centre_coordinates(coords, mask)
    coords_norm = apply_coordinate_normalisation(coords_centered, mask, stats)
    return _mask_flattened_coords(flatten_backbone(coords_norm), mask)


def apply_length_aware_radius_projection(
    x_flat: torch.Tensor,
    mask: torch.Tensor,
    stats: BackboneNormalizationStats,
    target_scale: float,
    strength: float,
    max_scale_step: float,
) -> torch.Tensor:
    """Expand residue positions when the C-alpha radius is below a conservative length-aware target.

    The projection scales residue C-alpha positions around the structure centre, but preserves
    each residue's internal atom offsets relative to its C-alpha. This avoids directly stretching
    N-CA, CA-C, and C-O bonds while still testing whether the global collapse problem can be
    reduced during sampling.
    """
    if strength <= 0.0 or target_scale <= 0.0 or max_scale_step <= 0.0:
        return x_flat

    mask = mask.to(device=x_flat.device)
    coords = normalized_flat_to_centered_angstrom(x_flat, mask, stats)
    valid = mask.to(device=x_flat.device, dtype=coords.dtype)
    ca = coords[:, :, 1, :]
    residue_count = valid.sum(dim=1).clamp_min(1.0)
    rg = torch.sqrt(((ca.pow(2).sum(dim=-1) * valid).sum(dim=1) / residue_count).clamp_min(1e-8))
    target_rg = target_scale * torch.sqrt(residue_count)

    raw_scale = (target_rg / rg.clamp_min(1e-6)).clamp_min(1.0)
    step_scale = 1.0 + strength * (raw_scale - 1.0)
    step_scale = torch.minimum(step_scale, torch.full_like(step_scale, 1.0 + max_scale_step))

    local_offsets = coords - ca[:, :, None, :]
    ca_scaled = ca * step_scale[:, None, None]
    coords_scaled = ca_scaled[:, :, None, :] + local_offsets
    return centered_angstrom_to_normalized_flat(coords_scaled, mask, stats)


def apply_adjacent_ca_residue_projection(
    x_flat: torch.Tensor,
    mask: torch.Tensor,
    stats: BackboneNormalizationStats,
    target_distance: float = 3.80,
    strength: float = 0.05,
    iterations: int = 1,
) -> torch.Tensor:
    """Lightly correct adjacent C-alpha spacing by translating whole residues.

    Each residue is moved as a small rigid group, preserving intra-residue atom offsets.
    This is a heuristic local projection for sampling only, not a training objective.
    """
    if strength <= 0.0 or iterations <= 0:
        return x_flat

    mask = mask.to(device=x_flat.device)
    coords = normalized_flat_to_centered_angstrom(x_flat, mask, stats)
    valid_pair = (mask[:, :-1].bool() & mask[:, 1:].bool()).to(device=x_flat.device)

    for _ in range(iterations):
        ca = coords[:, :, 1, :]
        diff = ca[:, 1:, :] - ca[:, :-1, :]
        dist = torch.linalg.norm(diff, dim=-1).clamp_min(1e-6)
        unit = diff / dist.unsqueeze(-1)
        delta = 0.5 * strength * (target_distance - dist).unsqueeze(-1) * unit
        delta = delta * valid_pair.to(dtype=coords.dtype).unsqueeze(-1)
        shifts = torch.zeros_like(ca)
        shifts[:, :-1, :] -= delta
        shifts[:, 1:, :] += delta
        coords = coords + shifts[:, :, None, :]
        coords = centre_coordinates(coords, mask)

    return centered_angstrom_to_normalized_flat(coords, mask, stats)


@torch.no_grad()
def sample_backbone_with_projection_guidance(
    model: torch.nn.Module,
    schedule: dict[str, torch.Tensor],
    shape: tuple[int, int, int],
    mask: torch.Tensor,
    device: torch.device,
    stats: BackboneNormalizationStats,
    radius_target_scale: float = 0.0,
    radius_strength: float = 0.0,
    radius_max_scale_step: float = 0.0,
    local_projection_strength: float = 0.0,
    local_projection_iterations: int = 1,
    radius_start_timestep: int = V4H_RADIUS_GUIDANCE_START_TIMESTEP,
    radius_end_timestep: int = V4H_RADIUS_GUIDANCE_END_TIMESTEP,
    local_projection_max_timestep: int = V4H_LOCAL_PROJECTION_MAX_TIMESTEP,
) -> torch.Tensor:
    """DDPM sampling with optional lightweight projection after each reverse step."""

    if len(shape) != 3 or shape[2] != 12:
        raise ValueError('shape must be (B, L, 12).')
    batch_size, seq_len, _ = shape
    x = torch.randn(shape, device=device)
    mask = mask.to(device=device, dtype=torch.float32)
    betas = schedule['betas'].to(device)
    alphas = schedule['alphas'].to(device)
    alpha_bars = schedule['alpha_bars'].to(device)
    posterior_variance = schedule['posterior_variance'].to(device)
    timesteps = int(betas.shape[0] - 1)

    for timestep in range(timesteps, 0, -1):
        t = torch.full((batch_size,), timestep, device=device, dtype=torch.long)
        pred_noise = model(x, t, mask)
        alpha_t = alphas[timestep]
        beta_t = betas[timestep]
        alpha_bar_t = alpha_bars[timestep]
        mean = (x - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * pred_noise) / torch.sqrt(alpha_t)
        if timestep > 1:
            noise = torch.randn_like(x)
            variance = posterior_variance[timestep]
            x = mean + torch.sqrt(variance.clamp_min(1e-20)) * noise
        else:
            x = mean
        x = _mask_flattened_coords(x, mask)

        # Global projection first: encourage a conservative length-aware radius in the
        # high/mid-noise part of sampling, where broad shape is being decided.
        if radius_strength > 0.0 and radius_end_timestep <= timestep <= radius_start_timestep:
            x = apply_length_aware_radius_projection(
                x,
                mask,
                stats,
                target_scale=radius_target_scale,
                strength=radius_strength,
                max_scale_step=radius_max_scale_step,
            )

        # Local projection second: lightly repair adjacent C-alpha spacing in the lower-noise
        # part of sampling, where local backbone continuity should be refined.
        if local_projection_strength > 0.0 and timestep <= local_projection_max_timestep:
            x = apply_adjacent_ca_residue_projection(
                x,
                mask,
                stats,
                target_distance=GEOMETRY_TARGETS_ANGSTROM['adjacent_ca'],
                strength=local_projection_strength,
                iterations=local_projection_iterations,
            )
        x = _mask_flattened_coords(x, mask)

    return x


def sample_condition_with_projection_guidance(
    condition: dict[str, object],
    masks: torch.Tensor,
    real_coords: torch.Tensor,
    sample_batch_size: int = V4H_GUIDANCE_BATCH_SIZE,
) -> tuple[pd.DataFrame, torch.Tensor]:
    """Sample/evaluate one baseline or guided condition."""

    condition_name = str(condition['condition'])
    generated_batches = []
    generated_rows = []
    for start in range(0, masks.shape[0], sample_batch_size):
        end = min(start + sample_batch_size, masks.shape[0])
        batch_mask = masks[start:end].to(device)
        if condition_name == 'baseline_v4h':
            sampled_flat = sample_backbone(
                model,
                noise_schedule,
                shape=(batch_mask.shape[0], MAX_SEQ_LENGTH, 12),
                mask=batch_mask,
                device=device,
            )
        else:
            sampled_flat = sample_backbone_with_projection_guidance(
                model,
                noise_schedule,
                shape=(batch_mask.shape[0], MAX_SEQ_LENGTH, 12),
                mask=batch_mask,
                device=device,
                stats=normalization_stats,
                radius_target_scale=float(condition.get('radius_target_scale', 0.0)),
                radius_strength=float(condition.get('radius_strength', 0.0)),
                radius_max_scale_step=float(condition.get('radius_max_scale_step', 0.0)),
                local_projection_strength=float(condition.get('local_projection_strength', 0.0)),
                local_projection_iterations=int(condition.get('local_projection_iterations', 1)),
            )
        generated_coords = invert_coordinate_normalisation(unflatten_backbone(sampled_flat), normalization_stats).cpu()
        generated_batches.append(generated_coords)
        for local_index in range(generated_coords.shape[0]):
            index = start + local_index
            generated_rows.append({
                'condition': condition_name,
                'kind': 'generated',
                'index': index,
                **backbone_structure_summary(generated_coords[local_index], masks[index]),
                **nonlocal_ca_contact_diagnostics(generated_coords[local_index], masks[index]),
            })
    return pd.DataFrame(generated_rows), torch.cat(generated_batches, dim=0)


v4h_guidance_conditions = [
    {
        'condition': 'baseline_v4h',
        'description': 'Standard DDPM reverse sampling with no sampling-time projection.',
    },
    {
        'condition': 'radius_project_weak',
        'description': 'Conservative length-aware radius projection plus light low-noise adjacent-CA repair.',
        'radius_target_scale': 0.85,
        'radius_strength': 0.05,
        'radius_max_scale_step': 0.010,
        'local_projection_strength': 0.03,
        'local_projection_iterations': 1,
    },
    {
        'condition': 'radius_project_medium',
        'description': 'Moderate radius projection plus low-noise adjacent-CA repair.',
        'radius_target_scale': 0.95,
        'radius_strength': 0.06,
        'radius_max_scale_step': 0.015,
        'local_projection_strength': 0.05,
        'local_projection_iterations': 1,
    },
    {
        'condition': 'radius_project_strong',
        'description': 'Stronger projection stress test; useful for checking whether expansion damages local geometry.',
        'radius_target_scale': 1.05,
        'radius_strength': 0.07,
        'radius_max_scale_step': 0.020,
        'local_projection_strength': 0.07,
        'local_projection_iterations': 1,
    },
]

v4h_guidance_real_coords, v4h_guidance_masks = collect_conditioning_masks(validation_loader, V4H_GUIDANCE_SAMPLE_COUNT)
real_guidance_rows = []
for index in range(v4h_guidance_real_coords.shape[0]):
    real_guidance_rows.append({
        'condition': 'real_validation',
        'kind': 'real',
        'index': index,
        **backbone_structure_summary(v4h_guidance_real_coords[index], v4h_guidance_masks[index]),
        **nonlocal_ca_contact_diagnostics(v4h_guidance_real_coords[index], v4h_guidance_masks[index]),
    })
v4h_guidance_real_df = pd.DataFrame(real_guidance_rows)

condition_frames = []
condition_sampled_coords: dict[str, torch.Tensor] = {}
for condition in v4h_guidance_conditions:
    condition_df, condition_coords = sample_condition_with_projection_guidance(
        condition,
        v4h_guidance_masks,
        v4h_guidance_real_coords,
        sample_batch_size=V4H_GUIDANCE_BATCH_SIZE,
    )
    condition_frames.append(condition_df)
    condition_sampled_coords[str(condition['condition'])] = condition_coords

v4h_guidance_generated_df = pd.concat(condition_frames, ignore_index=True)
v4h_guidance_all_df = pd.concat([v4h_guidance_real_df, v4h_guidance_generated_df], ignore_index=True)

real_rg_median_guidance = float(v4h_guidance_real_df['radius_of_gyration'].median())
collapse_radius_threshold_guidance = 0.5 * real_rg_median_guidance
v4h_guidance_generated_df['collapse_flag'] = v4h_guidance_generated_df['radius_of_gyration'] < collapse_radius_threshold_guidance
v4h_guidance_generated_df['poor_ca_band_flag'] = v4h_guidance_generated_df['fraction_adjacent_ca_in_band'] < 0.8

v4h_guidance_summary = v4h_guidance_generated_df.groupby('condition', as_index=False).agg(
    n_structures=('index', 'count'),
    mean_adjacent_ca=('mean_adjacent_ca', 'mean'),
    mean_fraction_adjacent_ca_in_band=('fraction_adjacent_ca_in_band', 'mean'),
    mean_radius_of_gyration=('radius_of_gyration', 'mean'),
    collapse_count=('collapse_flag', 'sum'),
    poor_ca_band_count=('poor_ca_band_flag', 'sum'),
    mean_nonlocal_ca_distance=('mean_nonlocal_ca_distance', 'mean'),
    mean_fraction_nonlocal_ca_below_8A=('fraction_nonlocal_ca_below_8A', 'mean'),
    mean_fraction_nonlocal_ca_below_10A=('fraction_nonlocal_ca_below_10A', 'mean'),
    mean_max_pairwise_ca_distance=('max_pairwise_ca_distance', 'mean'),
    mean_end_to_end_ca_distance=('end_to_end_ca_distance', 'mean'),
)

v4h_real_guidance_summary = pd.DataFrame([{
    'condition': 'real_validation',
    'n_structures': len(v4h_guidance_real_df),
    'mean_adjacent_ca': v4h_guidance_real_df['mean_adjacent_ca'].mean(),
    'mean_fraction_adjacent_ca_in_band': v4h_guidance_real_df['fraction_adjacent_ca_in_band'].mean(),
    'mean_radius_of_gyration': v4h_guidance_real_df['radius_of_gyration'].mean(),
    'collapse_count': 0,
    'poor_ca_band_count': 0,
    'mean_nonlocal_ca_distance': v4h_guidance_real_df['mean_nonlocal_ca_distance'].mean(),
    'mean_fraction_nonlocal_ca_below_8A': v4h_guidance_real_df['fraction_nonlocal_ca_below_8A'].mean(),
    'mean_fraction_nonlocal_ca_below_10A': v4h_guidance_real_df['fraction_nonlocal_ca_below_10A'].mean(),
    'mean_max_pairwise_ca_distance': v4h_guidance_real_df['max_pairwise_ca_distance'].mean(),
    'mean_end_to_end_ca_distance': v4h_guidance_real_df['end_to_end_ca_distance'].mean(),
}])
v4h_guidance_summary_with_real = pd.concat([v4h_real_guidance_summary, v4h_guidance_summary], ignore_index=True)

v4h_guidance_config_df = pd.DataFrame(v4h_guidance_conditions)
v4h_guidance_config_df['radius_guidance_start_timestep'] = V4H_RADIUS_GUIDANCE_START_TIMESTEP
v4h_guidance_config_df['radius_guidance_end_timestep'] = V4H_RADIUS_GUIDANCE_END_TIMESTEP
v4h_guidance_config_df['local_projection_max_timestep'] = V4H_LOCAL_PROJECTION_MAX_TIMESTEP
v4h_guidance_config_df['sample_count'] = V4H_GUIDANCE_SAMPLE_COUNT
v4h_guidance_config_df['sample_batch_size'] = V4H_GUIDANCE_BATCH_SIZE

for frame in [v4h_guidance_summary_with_real, v4h_guidance_config_df]:
    display(frame)

save_table_artifact(v4h_guidance_real_df, 'v4h_guidance_real_eval_metrics')
save_table_artifact(v4h_guidance_generated_df, 'v4h_guidance_generated_eval_metrics')
save_table_artifact(v4h_guidance_all_df, 'v4h_guidance_all_eval_metrics')
save_table_artifact(v4h_guidance_summary_with_real, 'v4h_guidance_comparison_summary')
save_table_artifact(v4h_guidance_config_df, 'v4h_guidance_condition_config')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_frame = v4h_guidance_summary_with_real.copy()
axes[0].bar(plot_frame['condition'], plot_frame['mean_radius_of_gyration'])
axes[0].set_title('Mean radius of gyration')
axes[0].tick_params(axis='x', rotation=45)
axes[1].bar(plot_frame['condition'], plot_frame['mean_fraction_adjacent_ca_in_band'])
axes[1].set_title('Adjacent CA in-band fraction')
axes[1].tick_params(axis='x', rotation=45)
axes[2].bar(plot_frame['condition'], plot_frame['mean_fraction_nonlocal_ca_below_8A'])
axes[2].set_title('Nonlocal CA fraction below 8 A')
axes[2].tick_params(axis='x', rotation=45)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'v4h_guidance_condition_comparison.png', dpi=150)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics_to_plot = [
    ('mean_adjacent_ca', 'Mean adjacent CA distance'),
    ('fraction_adjacent_ca_in_band', 'Adjacent CA in-band fraction'),
    ('radius_of_gyration', 'Radius of gyration'),
]
for ax, (metric, label) in zip(axes, metrics_to_plot):
    for kind, frame in v4h_comparison_df.groupby('kind'):
        ax.hist(frame[metric].dropna(), bins=12, alpha=0.6, label=kind)
    ax.set_title(label)
    ax.set_xlabel(metric)
    ax.set_ylabel('Count')
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'v4h_structural_metric_distributions.png', dpi=150)
plt.show()



### Reverse-trajectory collapse diagnostics

This section samples selected reverse-diffusion states, reconstructs clean structures from those intermediate states, and measures when global compactness starts to emerge. These diagnostics are useful for comparing standard V4-style sampling against the guided/projection variants.


In [ ]:
@torch.no_grad()
def sample_backbone_with_trajectory(
    model: torch.nn.Module,
    schedule: dict[str, torch.Tensor],
    shape: tuple[int, int, int],
    mask: torch.Tensor,
    device: torch.device,
    trajectory_timesteps: tuple[int, ...] = (100, 75, 50, 25, 10, 0),
    t_start: int | None = None,
) -> tuple[torch.Tensor, dict[int, torch.Tensor]]:
    """Sample a backbone and store selected intermediate x_t tensors during reverse diffusion.

    ``t_start`` is a sampling-only diagnostic control. When left as ``None`` it uses the
    full schedule. When set to a smaller value, sampling starts from Gaussian noise at
    that reverse timestep and skips the most extreme high-noise part of the schedule.
    This is useful for checking whether V4h's failure is concentrated near the near-zero
    terminal-SNR endpoint.
    """

    if len(shape) != 3 or shape[2] != 12:
        raise ValueError('shape must be (B, L, 12).')
    batch_size, seq_len, _ = shape
    x = torch.randn(shape, device=device)
    mask = mask.to(device=device, dtype=torch.float32)
    betas = schedule['betas'].to(device)
    alphas = schedule['alphas'].to(device)
    alpha_bars = schedule['alpha_bars'].to(device)
    posterior_variance = schedule['posterior_variance'].to(device)
    timesteps = int(betas.shape[0] - 1)
    if t_start is None:
        t_start = timesteps
    t_start = int(min(max(t_start, 1), timesteps))

    requested_timesteps = {int(timestep) for timestep in trajectory_timesteps if int(timestep) <= t_start}
    trajectory_states: dict[int, torch.Tensor] = {}
    if t_start in requested_timesteps:
        trajectory_states[t_start] = x.detach().cpu()

    for timestep in range(t_start, 0, -1):
        t = torch.full((batch_size,), timestep, device=device, dtype=torch.long)
        pred_noise = model(x, t, mask)
        alpha_t = alphas[timestep]
        beta_t = betas[timestep]
        alpha_bar_t = alpha_bars[timestep]
        mean = (x - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * pred_noise) / torch.sqrt(alpha_t)
        if timestep > 1:
            noise = torch.randn_like(x)
            variance = posterior_variance[timestep]
            x = mean + torch.sqrt(variance.clamp_min(1e-20)) * noise
        else:
            x = mean
        x = x * mask.unsqueeze(-1)
        stored_timestep = timestep - 1
        if stored_timestep in requested_timesteps:
            trajectory_states[stored_timestep] = x.detach().cpu()

    return x, trajectory_states


def estimate_clean_coords_from_trajectory_state(
    model: torch.nn.Module,
    trajectory_state: torch.Tensor,
    timestep: int,
    mask: torch.Tensor,
    stats: BackboneNormalizationStats,
    schedule: dict[str, torch.Tensor],
    device: torch.device,
) -> torch.Tensor:
    """Estimate clean coordinates from a stored reverse-diffusion state."""

    batch_size = trajectory_state.shape[0]
    x_t = trajectory_state.to(device)
    mask = mask.to(device)
    if timestep == 0:
        coords_est = invert_coordinate_normalisation(unflatten_backbone(x_t), stats).cpu()
        return coords_est
    t = torch.full((batch_size,), timestep, device=device, dtype=torch.long)
    pred_noise = model(x_t, t, mask)
    x0_est = differentiable_predict_x0(x_t, t, pred_noise, schedule['alpha_bars'])
    coords_est = invert_coordinate_normalisation(unflatten_backbone(x0_est), stats).cpu()
    return coords_est


def evaluate_reverse_trajectory_collapse(
    model: torch.nn.Module,
    masks: torch.Tensor,
    stats: BackboneNormalizationStats,
    schedule: dict[str, torch.Tensor],
    device: torch.device,
    trajectory_timesteps: tuple[int, ...] = (100, 75, 50, 25, 10, 0),
    sample_batch_size: int = 4,
    t_start: int | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Evaluate collapse diagnostics across selected reverse-diffusion timesteps."""

    model.eval()
    per_structure_rows: list[dict[str, object]] = []
    for start in range(0, masks.shape[0], sample_batch_size):
        end = min(start + sample_batch_size, masks.shape[0])
        batch_mask = masks[start:end].to(device)
        _, trajectory_states = sample_backbone_with_trajectory(
            model,
            schedule,
            shape=(batch_mask.shape[0], MAX_SEQ_LENGTH, 12),
            mask=batch_mask,
            device=device,
            trajectory_timesteps=trajectory_timesteps,
            t_start=t_start,
        )
        for timestep in sorted(trajectory_states.keys(), reverse=True):
            coords_est = estimate_clean_coords_from_trajectory_state(
                model,
                trajectory_states[timestep],
                timestep,
                batch_mask,
                stats,
                schedule,
                device,
            )
            for local_index in range(coords_est.shape[0]):
                diagnostics = backbone_structure_summary(coords_est[local_index], batch_mask[local_index].cpu())
                diagnostics.update(nonlocal_ca_contact_diagnostics(coords_est[local_index], batch_mask[local_index].cpu()))
                per_structure_rows.append(
                    {
                        'sample_index': start + local_index,
                        'timestep': timestep,
                        **diagnostics,
                    }
                )
    diagnostics_df = pd.DataFrame(per_structure_rows)
    summary_df = diagnostics_df.groupby('timestep', as_index=False).agg(
        n_structures=('sample_index', 'count'),
        mean_radius_of_gyration=('radius_of_gyration', 'mean'),
        mean_fraction_adjacent_ca_in_band=('fraction_adjacent_ca_in_band', 'mean'),
        mean_fraction_nonlocal_ca_below_8A=('fraction_nonlocal_ca_below_8A', 'mean'),
        mean_fraction_nonlocal_ca_below_10A=('fraction_nonlocal_ca_below_10A', 'mean'),
        mean_nonlocal_ca_distance=('mean_nonlocal_ca_distance', 'mean'),
        mean_max_pairwise_ca_distance=('max_pairwise_ca_distance', 'mean'),
        mean_end_to_end_ca_distance=('end_to_end_ca_distance', 'mean'),
    ).sort_values('timestep', ascending=False)
    return diagnostics_df, summary_df


def ca_distance_matrix(coords: torch.Tensor, mask: torch.Tensor) -> np.ndarray:
    """Return a valid-residue C-alpha distance matrix for one structure."""

    if coords.ndim == 4:
        coords = coords[0]
    if coords.ndim != 3 or coords.shape[-2:] != (4, 3):
        raise ValueError('coords must have shape (L, 4, 3).')
    if mask.ndim != 1:
        raise ValueError('mask must have shape (L,).')

    mask_bool = mask.bool().cpu().numpy()
    ca = coords[:, 1, :].detach().cpu().numpy()
    distance_matrix = np.full((coords.shape[0], coords.shape[0]), np.nan, dtype=np.float32)
    valid = mask_bool[:, None] & mask_bool[None, :]
    if valid.any():
        diff = ca[:, None, :] - ca[None, :, :]
        distance_matrix[valid] = np.linalg.norm(diff, axis=-1)[valid]
    return distance_matrix


def ca_contact_matrix(coords: torch.Tensor, mask: torch.Tensor, threshold: float = 8.0) -> np.ndarray:
    """Return a binary valid-residue C-alpha contact map for one structure."""

    if coords.ndim == 4:
        coords = coords[0]
    if coords.ndim != 3 or coords.shape[-2:] != (4, 3):
        raise ValueError('coords must have shape (L, 4, 3).')
    if mask.ndim != 1:
        raise ValueError('mask must have shape (L,).')

    distance_matrix = ca_distance_matrix(coords, mask)
    contact_matrix = np.zeros_like(distance_matrix, dtype=np.float32)
    valid = np.isfinite(distance_matrix)
    contact_matrix[valid] = (distance_matrix[valid] < threshold).astype(np.float32)
    return contact_matrix


def save_ca_map_comparisons(
    real_coords: torch.Tensor,
    generated_coords: torch.Tensor,
    masks: torch.Tensor,
    max_examples: int = 3,
) -> None:
    """Save compact real-versus-generated CA distance/contact comparisons."""

    example_count = min(max_examples, real_coords.shape[0], generated_coords.shape[0])
    if example_count == 0:
        return

    real_distance_mats = [ca_distance_matrix(real_coords[i], masks[i].cpu()) for i in range(example_count)]
    generated_distance_mats = [ca_distance_matrix(generated_coords[i], masks[i].cpu()) for i in range(example_count)]
    real_contact_mats = [ca_contact_matrix(real_coords[i], masks[i].cpu(), threshold=8.0) for i in range(example_count)]
    generated_contact_mats = [ca_contact_matrix(generated_coords[i], masks[i].cpu(), threshold=8.0) for i in range(example_count)]

    finite_max = [float(np.nanmax(matrix)) for matrix in [*real_distance_mats, *generated_distance_mats] if np.isfinite(matrix).any()]
    max_distance = max(finite_max) if finite_max else 1.0

    fig, axes = plt.subplots(2, example_count, figsize=(5 * example_count, 8), squeeze=False)
    for index in range(example_count):
        real_img = np.ma.masked_invalid(real_distance_mats[index])
        gen_img = np.ma.masked_invalid(generated_distance_mats[index])
        for ax, img, title in [
            (axes[0, index], real_img, f'Real {index}'),
            (axes[1, index], gen_img, f'Generated {index}'),
        ]:
            plot = ax.imshow(img, origin='lower', cmap='viridis', vmin=0.0, vmax=max_distance)
            ax.set_title(title)
            ax.set_xlabel('Residue index')
            ax.set_ylabel('Residue index')
        fig.colorbar(plot, ax=axes[:, index].tolist(), shrink=0.7)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / 'v4h_ca_distance_maps_real_vs_generated.png', dpi=150)
    plt.show()

    fig, axes = plt.subplots(2, example_count, figsize=(5 * example_count, 8), squeeze=False)
    for index in range(example_count):
        for ax, img, title in [
            (axes[0, index], real_contact_mats[index], f'Real {index}'),
            (axes[1, index], generated_contact_mats[index], f'Generated {index}'),
        ]:
            plot = ax.imshow(img, origin='lower', cmap='magma', vmin=0.0, vmax=1.0)
            ax.set_title(title)
            ax.set_xlabel('Residue index')
            ax.set_ylabel('Residue index')
        fig.colorbar(plot, ax=axes[:, index].tolist(), shrink=0.7)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / 'v4h_ca_contact_maps_real_vs_generated_thr8.png', dpi=150)
    plt.show()
v4h_nonlocal_ca_diagnostics_df = v4h_comparison_df[[
    'kind',
    'index',
    'n_residues',
    'mean_adjacent_ca',
    'fraction_adjacent_ca_in_band',
    'radius_of_gyration',
    'mean_nonlocal_ca_distance',
    'median_nonlocal_ca_distance',
    'fraction_nonlocal_ca_below_8A',
    'fraction_nonlocal_ca_below_10A',
    'max_pairwise_ca_distance',
    'end_to_end_ca_distance',
]].copy()

v4h_nonlocal_ca_summary = v4h_nonlocal_ca_diagnostics_df.groupby('kind', as_index=False).agg(
    n_structures=('index', 'count'),
    mean_radius_of_gyration=('radius_of_gyration', 'mean'),
    mean_adjacent_ca=('mean_adjacent_ca', 'mean'),
    mean_fraction_adjacent_ca_in_band=('fraction_adjacent_ca_in_band', 'mean'),
    mean_nonlocal_ca_distance=('mean_nonlocal_ca_distance', 'mean'),
    mean_median_nonlocal_ca_distance=('median_nonlocal_ca_distance', 'mean'),
    mean_fraction_nonlocal_ca_below_8A=('fraction_nonlocal_ca_below_8A', 'mean'),
    mean_fraction_nonlocal_ca_below_10A=('fraction_nonlocal_ca_below_10A', 'mean'),
    mean_max_pairwise_ca_distance=('max_pairwise_ca_distance', 'mean'),
    mean_end_to_end_ca_distance=('end_to_end_ca_distance', 'mean'),
)

display(v4h_nonlocal_ca_diagnostics_df.head())
display(v4h_nonlocal_ca_summary)
save_table_artifact(v4h_nonlocal_ca_diagnostics_df, 'v4h_nonlocal_ca_diagnostics')
save_table_artifact(v4h_nonlocal_ca_summary, 'v4h_nonlocal_ca_summary')

V4H_TRAJECTORY_SAMPLE_COUNT = min(8, len(validation_dataset))
V4H_TRAJECTORY_TIMESTEPS = (100, 75, 50, 25, 10, 0)
v4h_trajectory_real_coords, v4h_trajectory_masks = collect_conditioning_masks(validation_loader, V4H_TRAJECTORY_SAMPLE_COUNT)
v4h_trajectory_diagnostics_df, v4h_trajectory_summary_df = evaluate_reverse_trajectory_collapse(
    model,
    v4h_trajectory_masks,
    normalization_stats,
    noise_schedule,
    device,
    trajectory_timesteps=V4H_TRAJECTORY_TIMESTEPS,
    sample_batch_size=4,
)

display(v4h_trajectory_diagnostics_df.head())
display(v4h_trajectory_summary_df)
save_table_artifact(v4h_trajectory_diagnostics_df, 'v4h_reverse_trajectory_collapse_diagnostics')
save_table_artifact(v4h_trajectory_summary_df, 'v4h_reverse_trajectory_collapse_summary')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
trajectory_summary_sorted = v4h_trajectory_summary_df.sort_values('timestep', ascending=False)
axes[0].plot(trajectory_summary_sorted['timestep'], trajectory_summary_sorted['mean_radius_of_gyration'], marker='o')
axes[1].plot(trajectory_summary_sorted['timestep'], trajectory_summary_sorted['mean_fraction_nonlocal_ca_below_8A'], marker='o')
axes[1].plot(trajectory_summary_sorted['timestep'], trajectory_summary_sorted['mean_fraction_adjacent_ca_in_band'], marker='s', linestyle='--')
axes[2].plot(trajectory_summary_sorted['timestep'], trajectory_summary_sorted['mean_nonlocal_ca_distance'], marker='o')
axes[2].plot(trajectory_summary_sorted['timestep'], trajectory_summary_sorted['mean_end_to_end_ca_distance'], marker='s', linestyle='--')
axes[0].set_xlabel('Reverse timestep')
axes[0].set_ylabel('Radius of gyration')
axes[0].set_title('Trajectory compactness')
axes[1].set_xlabel('Reverse timestep')
axes[1].set_ylabel('Fraction')
axes[1].set_title('Nonlocal contacts and local CA banding')
axes[2].set_xlabel('Reverse timestep')
axes[2].set_ylabel('Distance (A)')
axes[2].set_title('Nonlocal distance scale')
axes[0].legend(['Rg'])
axes[1].legend(['nonlocal < 8 A', 'adjacent CA in band'])
axes[2].legend(['mean nonlocal', 'end-to-end'])
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'v4h_reverse_trajectory_collapse_diagnostics.png', dpi=150)
plt.show()

save_ca_map_comparisons(v4h_real_coords[:3], v4h_sampled_coords[:3], v4h_sample_masks[:3])



## 10. V4h interpretation guide

V4d, V4e, and V4h showed that training-time nonlocal losses can affect the collapse axis, but they did not cleanly beat the V4 reference. V4h therefore tests a different idea: keep the best V4-style trained model and intervene during sampling.

The desired outcome is not just a larger radius of gyration. A good guided sample should improve global compactness while preserving local backbone validity. The primary comparison should therefore use collapse count, radius of gyration, nonlocal C-alpha contact fractions, adjacent C-alpha in-band fraction, and bond-length summaries together.

If guided sampling improves radius/nonlocal contacts but damages adjacent C-alpha geometry, it should be treated as a useful failure mode rather than a final model. If it improves global compactness while preserving local geometry close to V4, then sampling-time guidance is the most promising next direction.


### V4h start-timestep diagnostic

This diagnostic reuses the trained V4h checkpoint and changes only the sampling start point. It does **not** retrain the model. The question is whether V4h explodes only when sampling starts from the most extreme near-pure-noise endpoint (`t_start=100`), or whether it remains unstable even when the highest-noise region is skipped (`t_start=90`, `75`, or `50`).


In [ ]:

# V4h start-timestep diagnostic.
# This is a sampling-only test. It reuses the trained V4h model/checkpoint and changes only where reverse sampling starts.

V4H_TSTART_VALUES = tuple(
    int(value.strip())
    for value in os.environ.get('BACKBONE_DIFFUSION_V4H_TSTART_VALUES', '100,90,75,50').split(',')
    if value.strip()
)
V4H_TSTART_SAMPLE_COUNT = int(os.environ.get('BACKBONE_DIFFUSION_V4H_TSTART_SAMPLE_COUNT', str(min(8, len(validation_dataset)))))
V4H_TSTART_BATCH_SIZE = int(os.environ.get('BACKBONE_DIFFUSION_V4H_TSTART_BATCH_SIZE', '4'))

print('V4h start-timestep diagnostic values:', V4H_TSTART_VALUES)
print('V4h start-timestep sample count:', V4H_TSTART_SAMPLE_COUNT)

v4h_tstart_real_coords, v4h_tstart_masks = collect_conditioning_masks(validation_loader, V4H_TSTART_SAMPLE_COUNT)

v4h_tstart_diagnostics_frames: list[pd.DataFrame] = []
v4h_tstart_summary_frames: list[pd.DataFrame] = []

for t_start in V4H_TSTART_VALUES:
    trajectory_timesteps = tuple(sorted({0, 10, 25, 50, 75, 90, 100, int(t_start)} & set(range(0, int(t_start) + 1)), reverse=True))
    diagnostics_df, summary_df = evaluate_reverse_trajectory_collapse(
        model,
        v4h_tstart_masks,
        normalization_stats,
        noise_schedule,
        device,
        trajectory_timesteps=trajectory_timesteps,
        sample_batch_size=V4H_TSTART_BATCH_SIZE,
        t_start=int(t_start),
    )
    diagnostics_df.insert(0, 't_start', int(t_start))
    summary_df.insert(0, 't_start', int(t_start))
    v4h_tstart_diagnostics_frames.append(diagnostics_df)
    v4h_tstart_summary_frames.append(summary_df)

v4h_tstart_diagnostics_df = pd.concat(v4h_tstart_diagnostics_frames, ignore_index=True)
v4h_tstart_trajectory_summary_df = pd.concat(v4h_tstart_summary_frames, ignore_index=True)

# Final generated structures are represented by timestep 0 for each t_start.
v4h_tstart_final_df = v4h_tstart_diagnostics_df[v4h_tstart_diagnostics_df['timestep'] == 0].copy()
real_radius_median_tstart = float(
    pd.Series([
        backbone_structure_summary(v4h_tstart_real_coords[i], v4h_tstart_masks[i])['radius_of_gyration']
        for i in range(v4h_tstart_real_coords.shape[0])
    ]).median()
)
collapse_radius_threshold_tstart = 0.5 * real_radius_median_tstart
v4h_tstart_final_df['collapse_flag'] = v4h_tstart_final_df['radius_of_gyration'] < collapse_radius_threshold_tstart
v4h_tstart_final_df['poor_ca_band_flag'] = v4h_tstart_final_df['fraction_adjacent_ca_in_band'] < 0.8

v4h_tstart_final_summary_df = v4h_tstart_final_df.groupby('t_start', as_index=False).agg(
    sample_count=('sample_index', 'count'),
    mean_radius_of_gyration=('radius_of_gyration', 'mean'),
    median_radius_of_gyration=('radius_of_gyration', 'median'),
    mean_adjacent_ca=('mean_adjacent_ca', 'mean'),
    mean_fraction_adjacent_ca_in_band=('fraction_adjacent_ca_in_band', 'mean'),
    mean_nonlocal_ca_distance=('mean_nonlocal_ca_distance', 'mean'),
    mean_fraction_nonlocal_ca_below_8A=('fraction_nonlocal_ca_below_8A', 'mean'),
    mean_fraction_nonlocal_ca_below_10A=('fraction_nonlocal_ca_below_10A', 'mean'),
    mean_max_pairwise_ca_distance=('max_pairwise_ca_distance', 'mean'),
    mean_end_to_end_ca_distance=('end_to_end_ca_distance', 'mean'),
    collapse_count=('collapse_flag', 'sum'),
    poor_ca_band_count=('poor_ca_band_flag', 'sum'),
).sort_values('t_start', ascending=False)

v4h_tstart_final_summary_df['real_radius_median'] = real_radius_median_tstart
v4h_tstart_final_summary_df['collapse_radius_threshold'] = collapse_radius_threshold_tstart
v4h_tstart_final_summary_df['collapse_fraction'] = v4h_tstart_final_summary_df['collapse_count'] / v4h_tstart_final_summary_df['sample_count']
v4h_tstart_final_summary_df['poor_ca_band_fraction'] = v4h_tstart_final_summary_df['poor_ca_band_count'] / v4h_tstart_final_summary_df['sample_count']

display(v4h_tstart_final_summary_df)
display(v4h_tstart_trajectory_summary_df.head(20))

save_table_artifact(v4h_tstart_diagnostics_df, 'v4h_tstart_reverse_trajectory_diagnostics')
save_table_artifact(v4h_tstart_trajectory_summary_df, 'v4h_tstart_reverse_trajectory_summary')
save_table_artifact(v4h_tstart_final_summary_df, 'v4h_tstart_final_sample_summary')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
plot_df = v4h_tstart_final_summary_df.sort_values('t_start', ascending=False)
axes[0].plot(plot_df['t_start'], plot_df['mean_radius_of_gyration'], marker='o')
axes[0].axhline(real_radius_median_tstart, linestyle='--', linewidth=1, label='real median Rg')
axes[0].set_title('Final radius by sampling start')
axes[0].set_xlabel('t_start')
axes[0].set_ylabel('Radius of gyration (A)')
axes[0].legend()

axes[1].plot(plot_df['t_start'], plot_df['mean_fraction_adjacent_ca_in_band'], marker='o')
axes[1].axhline(0.8, linestyle='--', linewidth=1, label='poor-CA threshold')
axes[1].set_title('Final adjacent C-alpha validity')
axes[1].set_xlabel('t_start')
axes[1].set_ylabel('Fraction in band')
axes[1].legend()

axes[2].plot(plot_df['t_start'], plot_df['mean_fraction_nonlocal_ca_below_8A'], marker='o', label='nonlocal < 8 A')
axes[2].plot(plot_df['t_start'], plot_df['mean_fraction_nonlocal_ca_below_10A'], marker='s', label='nonlocal < 10 A')
axes[2].set_title('Final nonlocal contact fractions')
axes[2].set_xlabel('t_start')
axes[2].set_ylabel('Fraction')
axes[2].legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'v4h_tstart_final_sample_summary.png', dpi=150)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for t_start in sorted(v4h_tstart_trajectory_summary_df['t_start'].unique(), reverse=True):
    frame = v4h_tstart_trajectory_summary_df[v4h_tstart_trajectory_summary_df['t_start'] == t_start].sort_values('timestep', ascending=False)
    axes[0].plot(frame['timestep'], frame['mean_radius_of_gyration'], marker='o', label=f't_start={t_start}')
    axes[1].plot(frame['timestep'], frame['mean_fraction_adjacent_ca_in_band'], marker='o', label=f't_start={t_start}')
    axes[2].plot(frame['timestep'], frame['mean_nonlocal_ca_distance'], marker='o', label=f't_start={t_start}')
axes[0].set_title('Trajectory radius')
axes[0].set_xlabel('Reverse timestep')
axes[0].set_ylabel('Radius of gyration (A)')
axes[1].set_title('Trajectory adjacent C-alpha validity')
axes[1].set_xlabel('Reverse timestep')
axes[1].set_ylabel('Fraction in band')
axes[2].set_title('Trajectory nonlocal distance scale')
axes[2].set_xlabel('Reverse timestep')
axes[2].set_ylabel('Distance (A)')
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'v4h_tstart_reverse_trajectory_comparison.png', dpi=150)
plt.show()
